# CS781 Phase 3 — Rhetorical Role prediction

## 0. Write `rr.py`, `rr_ed.py`, `ensemble_cv.py`

In [1]:
%%writefile rr.py

"""
rr.py  --  CS781 Phase 3 (Rhetorical Role prediction) on top of YOUR Phase-1 RoPE+MQA+SwiGLU decoder.

One run = one adaptation method.  Examples:
  python rr.py --method lorma   --data_dir DATA --tokenizer bpe_tokenizer.json --init model_step45000.pt --out runs/lorma
  python rr.py --method full    ... --out runs/full
  python rr.py --method kd      ... --teacher runs/full/best.pt --out runs/kd

Task: every sentence of a legal judgment gets one of 13 rhetorical-role labels (metric = macro-F1 over all sentences).
Model: documents are cut into <=512-token windows of consecutive sentences (each sentence starts with a <s> marker);
       the Phase-1 trunk runs over the window, token states are mean-pooled per sentence, and a linear head sees
       [pooled sentence ++ relative position in the document ++ CL/IT flag].

  A. Phase-1 tokenizer + model code (copied from your notebooks; edits marked [P2-EDIT]/[P3-EDIT])
  B. Data, windows, metric
  C. Adaptation methods: full / lora / lorma(+) / adalora / ia3 / prompt / prefix / kd
  D. Training, inference, prediction writer
"""
import argparse, hashlib, json, math, os, pickle, random, re, sys, time, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IGN = -100
PAD_ID = 256          
EOS_ID = 259          




class ByteLevelBPETokenizer:
    SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]

    def __init__(self):
        self.merges = {}
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        self.special_to_id = {}
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            tid = 256 + i
            self.special_to_id[tok] = tid
            self.id_to_bytes[tid] = tok.encode("utf-8")

    @property
    def vocab_size(self):
        return len(self.id_to_bytes)

    @staticmethod
    def _pair_counts(sequences):
        counts = {}
        for seq, freq in sequences:
            for pair in zip(seq, seq[1:]):
                counts[pair] = counts.get(pair, 0) + freq
        return counts

    @staticmethod
    def _apply_merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                out.append(new_id); i += 2
            else:
                out.append(seq[i]); i += 1
        return out

    def train(self, text, vocab_size, max_words=100000, verbose=False):   
        num_merges = vocab_size - self.vocab_size
        word_freq = {}
        for word in text.split():
            word_freq[word] = word_freq.get(word, 0) + 1
        sequences = [[list(w.encode("utf-8")), f] for w, f in word_freq.items()]
        next_id = self.vocab_size
        for _ in range(num_merges):
            counts = self._pair_counts(sequences)
            if not counts: break
            best = max(counts, key=counts.get)
            self.merges[best] = next_id
            self.id_to_bytes[next_id] = self.id_to_bytes[best[0]] + self.id_to_bytes[best[1]]
            for e in sequences: e[0] = self._apply_merge(e[0], best, next_id)
            next_id += 1
        return self

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": {f"{a},{b}": n for (a, b), n in self.merges.items()}}, f)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair_str, nid in data["merges"].items():
            a, b = (int(x) for x in pair_str.split(","))
            tok.merges[(a, b)] = nid
            tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


def rotate_half(x):
    x1 = x[..., ::2]; x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, q, k):
        T = q.size(-2)
        cos = self.cos[:, :, :T, :]; sin = self.sin[:, :, :T, :]
        return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class MultiQueryCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % kv_heads == 0
        self.n_heads, self.kv_heads = n_heads, kv_heads
        self.d_head = d_model // n_heads
        self.kv_dim = kv_heads * self.d_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.d_head, max_seq_len=max_seq_len, base=rope_base)
        self.dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.causal = True                                   
        self.prefix_kv = None                                

    def forward(self, x, pad_mask=None):                     
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        q, k = self.rope(q, k)
        P = 0
        if self.prefix_kv is not None:                       
            pk, pv = self.prefix_kv
            P = pk.size(0)
            pk = pk.permute(1, 0, 2).unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype)
            pv = pv.permute(1, 0, 2).unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype)
            k = torch.cat([pk, k], dim=2); v = torch.cat([pv, v], dim=2)
        if self.kv_heads != self.n_heads:
            r = self.n_heads // self.kv_heads
            k = k.repeat_interleave(r, dim=1); v = v.repeat_interleave(r, dim=1)
        dp = self.dropout if self.training else 0.0
        if self.causal and P == 0:                           
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=dp, is_causal=True)
        else:
            if self.causal:
                m = torch.ones(T, P + T, dtype=torch.bool, device=x.device)
                m[:, P:] = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
                m = m[None, None]
            elif pad_mask is None:
                m = None
            else:
                m = pad_mask if P == 0 else torch.cat([torch.ones(B, P, dtype=torch.bool, device=x.device), pad_mask], 1)
                m = m[:, None, None, :]                      
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=m, dropout_p=dp, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down(F.silu(self.gate(x)) * self.up(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base):
        super().__init__()
        self.norm_attn = RMSNorm(d_model)
        self.attn = MultiQueryCausalAttention(d_model, n_heads, kv_heads, max_seq_len, dropout, rope_base)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, pad_mask=None):                     
        x = x + self.attn(self.norm_attn(x), pad_mask)
        x = x + self.ffn(self.norm_ffn(x))
        return x


class RoPEMQADecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, kv_heads, d_ff, max_seq_len,
                 dropout=0.1, rope_base=10000.0, tie_weights=True):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base)
                                     for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.lm_head.weight = self.token_emb.weight


DEFAULT_LM_CFG = dict(vocab_size=12000, d_model=768, n_layers=12, n_heads=12, kv_heads=1,
                      d_ff=3072, max_seq_len=512, dropout=0.1, rope_base=10000.0)


def build_lm(cfg):
    c = {**DEFAULT_LM_CFG, **{k: v for k, v in cfg.items() if k in DEFAULT_LM_CFG}}
    return RoPEMQADecoderLM(**c, tie_weights=True), c


def load_phase1(path, n_layers=None, layer_map=None):
    """Load your Phase-1 weights.  Accepts model.pt ({state_dict, config}), checkpoint.pt ({model_state, run_config})
    or a raw state_dict (model_stepXXXXX.pt).  Optionally keeps only `layer_map` blocks (for the KD student)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = {}
    if isinstance(ck, dict) and "state_dict" in ck:
        sd, cfg = ck["state_dict"], ck.get("config", {})
    elif isinstance(ck, dict) and "model_state" in ck:
        sd = ck["model_state"]
        rc = ck.get("run_config", {})
        cfg = dict(vocab_size=rc.get("VOCAB_SIZE"), d_model=rc.get("D_MODEL"), n_layers=rc.get("N_LAYERS"),
                   n_heads=rc.get("N_HEADS"), d_ff=rc.get("D_FF"), max_seq_len=rc.get("BLOCK_SIZE"),
                   kv_heads=rc.get("KV_HEADS"), rope_base=rc.get("ROPE_BASE"))
        cfg = {k: v for k, v in cfg.items() if v is not None}
    else:
        sd = ck
    sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
    cfg["n_layers"] = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    cfg["vocab_size"], cfg["d_model"] = sd["token_emb.weight"].shape
    cfg["d_ff"] = sd["blocks.0.ffn.gate.weight"].shape[0]
    cfg["kv_heads"] = sd["blocks.0.attn.k_proj.weight"].shape[0] // (cfg["d_model"] // cfg.get("n_heads", 12))
    if layer_map is not None:                                
        new = {k: v for k, v in sd.items() if not k.startswith("blocks.")}
        for ni, oi in enumerate(layer_map):
            for k, v in sd.items():
                if k.startswith(f"blocks.{oi}."):
                    new[f"blocks.{ni}." + k.split(".", 2)[2]] = v
        sd, cfg["n_layers"] = new, len(layer_map)
    lm, cfg = build_lm(cfg)
    missing, unexpected = lm.load_state_dict(sd, strict=False)
    bad = [k for k in missing if k != "lm_head.weight"] + list(unexpected)
    assert not bad, f"checkpoint/architecture mismatch: {bad[:6]}"
    return lm, cfg







BOS_ID = 258          
POS_BINS = 20
PAD_MULTIPLE = 1      
PAD_CAP = 1 << 30     


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_vocab(data):
    """Same acceptance rules as evaluate_phase3.py: list | label->id | id->label.  Returns (labels, label2idx)."""
    if isinstance(data, list):
        labels = list(data)
    elif isinstance(data, dict) and data:
        vals = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            labels = [k for k, _ in sorted(data.items(), key=lambda kv: kv[1])]
        elif all(isinstance(v, str) for v in vals):
            labels = [v for _, v in sorted(data.items(), key=lambda kv: int(kv[0]) if str(kv[0]).isdigit() else kv[0])]
        else:
            raise ValueError("unsupported label_vocab.json format")
    else:
        raise ValueError("label_vocab.json must be a non-empty list or object")
    assert len(labels) == 13 and len(set(labels)) == 13, f"expected 13 unique labels, got {labels}"
    return labels, {l: i for i, l in enumerate(labels)}


def iter_raw_docs(obj):
    """Accepts  [ {id,text,labels}, ... ]   or   { id: {text,labels} }   or   { id: [sentences] }  -> (id, sentences, labels|None)."""
    def pick(d, names):
        for n in names:
            if n in d: return d[n]
        return None
    if isinstance(obj, dict) and "documents" in obj and isinstance(obj["documents"], (list, dict)):
        obj = obj["documents"]
    if isinstance(obj, list):
        for k, d in enumerate(obj):
            did = pick(d, ("id", "doc_id", "document_id", "name", "file"))
            did = str(did) if did is not None else str(k)
            yield did, pick(d, ("text", "sentences", "sents", "sentence")), pick(d, ("labels", "label", "roles", "rhetorical_roles", "annotations"))
    elif isinstance(obj, dict):
        for did, d in obj.items():
            if isinstance(d, dict):
                yield str(did), pick(d, ("text", "sentences", "sents", "sentence")), pick(d, ("labels", "label", "roles", "rhetorical_roles", "annotations"))
            elif isinstance(d, list):
                yield str(did), d, None
            else:
                raise ValueError(f"cannot parse document {did!r}: {type(d)}")
    else:
        raise ValueError(f"cannot parse top-level {type(obj)}")


def label_to_idx(v, label2idx, labels):
    if isinstance(v, bool): raise ValueError(v)
    if isinstance(v, int): return v
    if isinstance(v, str):
        if v in label2idx: return label2idx[v]
        if v.isdigit(): return int(v)
    raise ValueError(f"unknown label {v!r}")


class WordBPE:
    """Phase-1 encode(): whitespace words -> byte-BPE (min-rank merge loop) -> </s> after every word.  Words are cached."""
    def __init__(self, tok):
        self.tok, self.cache = tok, {}

    def word(self, w):
        r = self.cache.get(w)
        if r is None:
            tok = self.tok
            seq = list(w.encode("utf-8"))
            while len(seq) > 1:
                mergeable = [p for p in zip(seq, seq[1:]) if p in tok.merges]
                if not mergeable: break
                pair = min(mergeable, key=lambda p: tok.merges[p])
                seq = tok._apply_merge(seq, pair, tok.merges[pair])
            r = tuple(seq) + (EOS_ID,)
            self.cache[w] = r
        return r

    def sentence(self, s):
        out = []
        for w in str(s).split(): out.extend(self.word(w))
        return np.array(out if out else [EOS_ID], np.int32)


def build_docs(raw, dom, label2idx, labels, enc, tag=""):
    docs, bad = [], 0
    for did, sents, labs in raw:
        if not isinstance(sents, list): raise ValueError(f"{tag} doc {did}: sentences field is {type(sents)}")
        toks = [enc.sentence(s) for s in sents]
        y = None
        if labs is not None:
            if len(labs) != len(sents): bad += 1; labs = (list(labs) + ["None"] * len(sents))[:len(sents)]
            y = np.array([label_to_idx(l, label2idx, labels) for l in labs], np.int64)
        docs.append(dict(id=did, dom=dom, raw=toks, y=y, n=len(sents)))
    if bad: print(f"  WARNING {tag}: {bad} docs had len(labels) != len(sentences) (padded/truncated)")
    return docs


def finalize_docs(docs, max_sent, marker):
    """Truncate long sentences (keep head 2/3 + tail 1/3), add the sentence-start marker; fills d['sents'], d['lens']."""
    for d in docs:
        ss = []
        for t in d["raw"]:
            if len(t) > max_sent:
                h = (max_sent * 2) // 3
                t = np.concatenate([t[:h], t[len(t) - (max_sent - h):]])
            if marker: t = np.concatenate([np.array([BOS_ID], np.int32), t])
            ss.append(t)
        d["sents"], d["lens"] = ss, [len(t) for t in ss]
    return docs


def make_windows(lens, budget, core_frac=0.5, first_frac=None):
    """Partition the sentences into CORE chunks (<= core_frac*budget tokens) and extend each with left/right context sentences
    until `budget` tokens are used.  Returns [(lo, hi, a, b)] : window = sentences lo..hi, loss/prediction on core a..b."""
    n = len(lens); cb = max(1, int(budget * core_frac)); wins = []; i = 0
    cur = cb if first_frac is None else max(1, int(cb * first_frac))
    while i < n:
        j, tot = i, 0
        while j < n and (j == i or tot + lens[j] <= cur):
            tot += lens[j]; j += 1
        lo, hi, used = i, j, tot
        grow = True
        while grow:
            grow = False
            if lo > 0 and used + lens[lo - 1] <= budget: lo -= 1; used += lens[lo]; grow = True
            if hi < n and used + lens[hi] <= budget: used += lens[hi]; hi += 1; grow = True
        wins.append((lo, hi, i, j)); i = j; cur = cb
    return wins


def collate(items, docs, device, train_labels=True):
    """items: [(doc_idx, lo, hi, a, b)] -> tensors.  Labels only on the core a..b (IGN elsewhere)."""
    B = len(items)
    toks = [np.concatenate(docs[di]["sents"][lo:hi]) for di, lo, hi, a, b in items]
    L = max(len(t) for t in toks); S = max(hi - lo for _, lo, hi, _, _ in items)
    if PAD_MULTIPLE > 1: L = min(-(-L // PAD_MULTIPLE) * PAD_MULTIPLE, PAD_CAP)   
    ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    sid = torch.full((B, L), -1, dtype=torch.long)
    msk = torch.zeros((B, L), dtype=torch.bool)
    lab = torch.full((B, S), IGN, dtype=torch.long)
    pb = torch.zeros((B, S), dtype=torch.long)
    dom = torch.zeros((B,), dtype=torch.long)
    for k, ((di, lo, hi, a, b), t) in enumerate(zip(items, toks)):
        d = docs[di]
        ids[k, :len(t)] = torch.from_numpy(t.astype(np.int64)); msk[k, :len(t)] = True
        pos = 0
        for s in range(lo, hi):
            ln = d["lens"][s]; sid[k, pos:pos + ln] = s - lo; pos += ln
        idx = np.arange(lo, hi)
        pb[k, :hi - lo] = torch.from_numpy(np.minimum((idx * POS_BINS) // max(1, d["n"]), POS_BINS - 1).astype(np.int64))
        if train_labels and d["y"] is not None: lab[k, a - lo:b - lo] = torch.from_numpy(d["y"][a:b])
        dom[k] = d["dom"]
    return ids.to(device), sid.to(device), (None if msk.all() else msk.to(device)), pb.to(device), dom.to(device), lab.to(device)


def macro_prf(y_true, y_pred, K):
    """Identical to sklearn f1/precision/recall(labels=all K, average='macro', zero_division=0)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    cm = np.bincount(y_true * K + y_pred, minlength=K * K).reshape(K, K)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(1, 2 * tp + fp + fn), 0.0)
    p = np.where(tp + fp > 0, tp / np.maximum(1, tp + fp), 0.0)
    r = np.where(tp + fn > 0, tp / np.maximum(1, tp + fn), 0.0)
    return dict(F1=float(f1.mean()), P=float(p.mean()), R=float(r.mean()), per_f1=f1.tolist(), per_p=p.tolist(), per_r=r.tolist(),
                support=cm.sum(1).tolist())


def score_docs(docs, lps):
    y = np.concatenate([d["y"] for d in docs]); p = np.concatenate([lp.argmax(1) for lp in lps])
    return macro_prf(y, p, lps[0].shape[1])





class RRModel(nn.Module):
    """Phase-1 decoder trunk + sentence pooling + (pooled ++ position-in-document ++ CL/IT) -> 13-way linear head."""
    def __init__(self, lm, K, causal=False, pool="mean", head_dropout=0.1, pos_dim=32, dom_dim=8):
        super().__init__()
        self.token_emb, self.blocks, self.final_norm = lm.token_emb, lm.blocks, lm.final_norm
        D = self.token_emb.embedding_dim
        self.pool = pool
        self.pos_emb = nn.Embedding(POS_BINS, pos_dim); self.dom_emb = nn.Embedding(2, dom_dim)
        nn.init.normal_(self.pos_emb.weight, std=0.02); nn.init.normal_(self.dom_emb.weight, std=0.02)
        self.drop = nn.Dropout(head_dropout)
        self.head = nn.Linear(D + pos_dim + dom_dim, K)
        self.prompt = None          
        self.prefix = None          
        self.set_causal(causal)

    def set_causal(self, causal):
        self.causal = causal
        for b in self.blocks: b.attn.causal = causal

    def sentence_reps(self, h, sid, S):
        B, L, D = h.shape
        valid = sid >= 0
        flat = (torch.arange(B, device=h.device)[:, None] * S + sid.clamp(min=0))[valid]
        hv = h[valid].float()
        if self.pool == "mean":
            sums = torch.zeros(B * S, D, device=h.device).index_add_(0, flat, hv)
            cnt = torch.zeros(B * S, device=h.device).index_add_(0, flat, torch.ones_like(flat, dtype=torch.float))
            return (sums / cnt.clamp(min=1)[:, None]).view(B, S, D)
        pos = torch.arange(L, device=h.device)[None].expand(B, L)[valid]
        if self.pool == "last":
            sel = torch.full((B * S,), -1, device=h.device, dtype=torch.long).scatter_reduce_(0, flat, pos, "amax", include_self=True)
        else:   
            sel = torch.full((B * S,), L, device=h.device, dtype=torch.long).scatter_reduce_(0, flat, pos, "amin", include_self=True)
        sel = sel.clamp(0, L - 1)
        base = (torch.arange(B, device=h.device)[:, None] * L).expand(B, S).reshape(-1)
        return h.float().reshape(B * L, D)[base + sel].view(B, S, D)

    def trunk(self, ids, pad_mask):
        """token ids -> final-normed hidden states (B, L, D) for the REAL tokens (prompt positions stripped)."""
        B = ids.size(0)
        x = self.token_emb(ids); P = 0
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(x.dtype), x], 1)
            if pad_mask is not None: pad_mask = torch.cat([torch.ones(B, P, dtype=torch.bool, device=ids.device), pad_mask], 1)
        if self.prefix is not None:
            kv = self.prefix()
            for b, (pk, pv) in zip(self.blocks, kv): b.attn.prefix_kv = (pk, pv)
        for b in self.blocks: x = b(x, pad_mask)
        return self.final_norm(x)[:, P:]

    def forward(self, ids, sid, pad_mask, pos_bin, dom):
        B = ids.size(0); S = pos_bin.size(1)
        h = self.trunk(ids, pad_mask)
        rep = self.sentence_reps(h, sid, S)
        feat = torch.cat([rep, self.pos_emb(pos_bin), self.dom_emb(dom)[:, None, :].expand(B, S, -1)], -1)
        return self.head(self.drop(feat))


class PrefixEncoder(nn.Module):
    """Prefix tuning (Li & Liang 2021): P virtual key/value vectors per layer (kv_heads x d_head), optionally produced by a
    reparametrisation MLP (hidden>0) which is more stable to train than free parameters."""
    def __init__(self, n_layers, n_prefix, kv_heads, d_head, d_model, hidden):
        super().__init__()
        self.L, self.P, self.kvh, self.dh = n_layers, n_prefix, kv_heads, d_head
        out = n_layers * 2 * kv_heads * d_head
        if hidden > 0:
            self.emb = nn.Parameter(torch.randn(n_prefix, d_model) * 0.02)
            self.mlp = nn.Sequential(nn.Linear(d_model, hidden), nn.Tanh(), nn.Linear(hidden, out))
            self.direct = None
        else:
            self.direct = nn.Parameter(torch.randn(n_prefix, out) * 0.02)

    def forward(self):
        z = self.direct if self.direct is not None else self.mlp(self.emb)
        z = z.view(self.P, self.L, 2, self.kvh, self.dh)
        return [(z[:, l, 0], z[:, l, 1]) for l in range(self.L)]


class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) B A x   (Hu et al. 2021)"""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMAPlusLinear(nn.Module):
    """LoRMA_+ (Bihany et al. 2025): W' = ((alpha/r) B A + I) W0 -- multiplicative, output side.  h = W0 x ; y = h + s B (A h)."""
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))

    def forward(self, x):
        h = self.base(x)
        return h + self.s * F.linear(F.linear(h, self.A), self.B)


class AdaLoRALinear(nn.Module):
    """AdaLoRA (Zhang et al. 2023): dW = P diag(E) Q with a rank-r SVD-like triplet structure; triplets are pruned by importance.
    A = Q (r x in), B = P (out x r), E = singular values (r).  `mask` (r) is the pruning state set by AdaLoRACtl."""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.r, self.s = base, r, alpha / r
        self.E = nn.Parameter(torch.zeros(r))
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.B = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.register_buffer("mask", torch.ones(r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        z = F.linear(self.drop(x), self.A) * (self.E * self.mask)
        return self.base(x) + self.s * F.linear(z, self.B)


class AdaLoRACtl:
    """Importance-based rank allocation: sensitivity |w * grad| smoothed with EMA (beta1) times its uncertainty (beta2);
    triplet score = s(E_i) + mean_j s(A_ij) + mean_j s(B_ji).  Global budget follows a cubic schedule from init_r*M to target_r*M."""
    def __init__(self, model, init_r, target_r, tinit, tfinal, delta_t=10, beta1=0.85, beta2=0.85, orth=0.1):
        self.mods = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.n_total = sum(m.r for m in self.mods)
        self.init_budget, self.target_budget = self.n_total, int(round(target_r / init_r * self.n_total))
        self.tinit, self.tfinal, self.dt, self.b1, self.b2, self.orth = tinit, tfinal, delta_t, beta1, beta2, orth
        self.ipt, self.unc = {}, {}
        self.budget = self.init_budget

    def orth_loss(self):
        if not self.mods or self.orth <= 0: return 0.0
        tot = 0.0
        for m in self.mods:
            A, B = m.A.float(), m.B.float(); I = torch.eye(m.r, device=A.device)
            tot = tot + torch.linalg.matrix_norm(A @ A.T - I) + torch.linalg.matrix_norm(B.T @ B - I)
        return self.orth * tot / len(self.mods)

    def budget_at(self, step, total):
        if step <= self.tinit: return self.init_budget
        if step > total - self.tfinal: return self.target_budget
        c = 1 - (step - self.tinit) / max(1, total - self.tfinal - self.tinit)
        return int((self.init_budget - self.target_budget) * c ** 3 + self.target_budget)

    @torch.no_grad()
    def step(self, step, total):
        """Call after backward (gradients already unscaled), before the optimiser step."""
        final_start = total - self.tfinal
        if step > final_start + 1: return                                              
        for m in self.mods:
            for p in (m.E, m.A, m.B):
                if p.grad is None or not bool(torch.isfinite(p.grad).all()): return    
        for m in self.mods:
            for p in (m.E, m.A, m.B):
                k = id(p); ipt = (p * p.grad).abs()
                a = self.ipt.get(k); a = ipt.clone() if a is None else self.b1 * a + (1 - self.b1) * ipt
                u = self.unc.get(k); u = torch.zeros_like(ipt) if u is None else u
                u = self.b2 * u + (1 - self.b2) * (ipt - a).abs()
                self.ipt[k], self.unc[k] = a, u
        self.budget = self.budget_at(step, total)
        if not (step == final_start + 1 or (step > self.tinit and step % self.dt == 0 and step <= final_start)): return
        scores = []
        for m in self.mods:
            sc = lambda p: self.ipt[id(p)] * self.unc[id(p)]
            scores.append(sc(m.E) + sc(m.A).mean(1) + sc(m.B).mean(0))
        allv = torch.cat(scores)
        k = self.n_total - self.budget
        if k <= 0:
            for m in self.mods: m.mask.fill_(1.0)
            return
        thr = torch.kthvalue(allv, k).values
        for m, sc_ in zip(self.mods, scores): m.mask.copy_((sc_ > thr).float())

    def rank_report(self):
        act = [int(m.mask.sum().item()) for m in self.mods]
        return dict(total_active=sum(act), total_possible=self.n_total, mean_rank=sum(act) / max(1, len(act)))


class IA3Out(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x): return self.base(x) * self.l


class IA3In(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x): return self.base(x * self.l)


ATTN_T = ("q_proj", "k_proj", "v_proj", "out_proj")
FFN_T = ("gate", "up", "down")
HEAD_PREFIXES = ("head.", "pos_emb.", "dom_emb.")


def is_head_param(name): return name.startswith(HEAD_PREFIXES)


def apply_method(model, args, total_steps_hint=1000):
    """Freezes the trunk and installs the chosen adaptation.  Returns an optional AdaLoRACtl."""
    m = args.method
    for p in model.parameters(): p.requires_grad = False
    for n, p in model.named_parameters():
        if is_head_param(n): p.requires_grad = True                                
    tg = [t.strip() for t in args.targets.split(",") if t.strip()]

    def wrap_linears(factory, names):
        for b in model.blocks:
            for t in names:
                parent = b.attn if t in ATTN_T else b.ffn
                setattr(parent, t, factory(getattr(parent, t)))

    ctl = None
    if m in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    elif m == "lora":
        wrap_linears(lambda l: LoRALinear(l, args.rank, args.alpha, args.lora_dropout), tg)
    elif m == "lorma":
        wrap_linears(lambda l: LoRMAPlusLinear(l, args.rank, args.alpha), tg)
    elif m == "adalora":
        wrap_linears(lambda l: AdaLoRALinear(l, args.ada_init_r, args.alpha_ada, args.lora_dropout), tg)
        ctl = "pending"                                                           
    elif m == "ia3":
        for b in model.blocks:
            for t in [x.strip() for x in args.ia3_targets.split(",") if x.strip()]:
                parent = b.attn if t in ATTN_T else b.ffn
                lin = getattr(parent, t)
                setattr(parent, t, IA3In(lin) if t == "down" else IA3Out(lin))   
    elif m == "prompt":
        ids = torch.randint(260, model.token_emb.num_embeddings, (args.n_prompt,))
        model.prompt = nn.Parameter(model.token_emb.weight[ids].detach().clone())   
    elif m == "prefix":
        a0 = model.blocks[0].attn
        model.prefix = PrefixEncoder(len(model.blocks), args.n_prefix, a0.kv_heads, a0.d_head, model.token_emb.embedding_dim, args.prefix_hidden)
    else:
        raise ValueError(m)
    return ctl


def mark_new_params_trainable(model, before_ids):
    for p in model.parameters():
        if id(p) not in before_ids: p.requires_grad = True


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr





SPLITS = [("CL", 0), ("IT", 1)]


def load_split(data_dir, kind, enc, labels, label2idx):
    """kind in train/dev/test -> list of docs (CL first, then IT), in file order."""
    docs = []
    for dom_name, dom in SPLITS:
        p = os.path.join(data_dir, f"{dom_name}_{kind}.json")
        docs += build_docs(list(iter_raw_docs(load_json(p))), dom, label2idx, labels, enc, f"{dom_name}_{kind}")
    ids = [d["id"] for d in docs]
    assert len(set(ids)) == len(ids), f"duplicate document ids across CL/IT {kind}: {[i for i in ids if ids.count(i) > 1][:5]}"
    return docs


def bal_docs(docs, n):
    """first n docs, half CL / half IT (smoke tests)."""
    return docs[:n // 2] + [d for d in docs if d["dom"] == 1][:n - n // 2] if n else docs


def make_folds(docs, k, seed=0):
    """Deterministic, domain-stratified, rare-label-aware k-fold assignment.  Independent of file order and of --seed:
    docs of each domain are ranked by how many rare label types they contain (rarity = 1 / #docs containing the label), ties broken by a seeded shuffle,
    then dealt round-robin to the folds -> every fold gets a similar mix of CL/IT and of the rare roles (Dissent, PrecedentOverruled, Issue ...)."""
    K = max(int(max(d["y"].max() for d in docs)) + 1, 1)
    df = np.zeros(K)
    for d in docs: df[np.unique(d["y"])] += 1
    rare = 1.0 / np.maximum(df, 1)
    rng = np.random.RandomState(seed); fold = {}
    for dom in sorted({d["dom"] for d in docs}):
        ds = sorted([d for d in docs if d["dom"] == dom], key=lambda d: d["id"])
        tie = rng.permutation(len(ds))
        score = [float(rare[np.unique(d["y"])].sum()) for d in ds]
        order = sorted(range(len(ds)), key=lambda i: (-score[i], tie[i]))
        start = int(rng.randint(k))
        for r, i in enumerate(order): fold[ds[i]["id"]] = (r + start) % k
    return fold


def get_docs(args, enc, labels, label2idx, tok_path):
    key = hashlib.md5(open(tok_path, "rb").read()[:100000] + b"rr-v1").hexdigest()[:10]
    cache = args.cache or os.path.join(args.data_dir, f"rrcache_{key}.pkl")
    if os.path.exists(cache):
        with open(cache, "rb") as f: out = pickle.load(f)
        print(f"  loaded tokenised cache {cache}")
        return out
    out = {k: load_split(args.data_dir, k, enc, labels, label2idx) for k in ("train", "dev", "test")}
    tmp = f"{cache}.{os.getpid()}.tmp"
    with open(tmp, "wb") as f: pickle.dump(out, f)
    os.replace(tmp, cache)
    return out


def batch_forward(model, docs, items, device, train_labels=True):
    ids, sid, msk, pb, dom, lab = collate(items, docs, device, train_labels)
    return model(ids, sid, msk, pb, dom), lab


TTA_OFFSETS = (None, 0.5, 0.25, 0.75, 0.375, 0.625)      


@torch.no_grad()
def infer_docs(model, docs, budget, core_frac, device, amp=True, bs=16, n_tta=1):
    """Per-document log-probs (n_sent, K).  Every sentence is the CORE of exactly one window and gets context from its neighbours.
    n_tta > 1: repeat with the window boundaries shifted (different context for every sentence) and average the PROBABILITIES."""
    was = model.training; model.eval()
    K = model.head.out_features
    acc = [np.zeros((d["n"], K), np.float64) for d in docs]
    offs = TTA_OFFSETS[:max(1, n_tta)]
    for ff in offs:
        items = []
        for di, d in enumerate(docs):
            for lo, hi, a, b in make_windows(d["lens"], budget, core_frac, first_frac=ff):
                items.append((di, lo, hi, a, b, int(sum(d["lens"][lo:hi]))))
        items.sort(key=lambda t: t[-1])
        for i in range(0, len(items), bs):
            chunk = items[i:i + bs]
            with torch.autocast("cuda", enabled=amp):
                logits, _ = batch_forward(model, docs, [c[:5] for c in chunk], device, train_labels=False)
            lp = F.log_softmax(logits.float(), -1).cpu().numpy()
            for k, (di, lo, hi, a, b, _) in enumerate(chunk): acc[di][a:b] += np.exp(lp[k, a - lo:b - lo].astype(np.float64))
    if was: model.train()
    return [np.log(np.maximum(a / len(offs), 1e-30)).astype(np.float32) for a in acc]


def lr_at(step, total, warm, base):
    if step < warm: return base * (step + 1) / max(1, warm)
    p = (step - warm) / max(1, total - warm)
    return base * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))


def meta_of(args, lm_cfg, labels, model):
    return dict(method=args.method, causal=model.causal, pool=model.pool, marker=args.marker, max_sent=args.max_sent,
                lm_cfg=lm_cfg, labels=labels, n_layers=len(model.blocks))


def load_teacher(path, device):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = ck["meta"]; assert meta["method"] == "full", "KD teacher must be a --method full run (best.pt)"
    lm, _ = build_lm(meta["lm_cfg"])
    model = RRModel(lm, len(meta["labels"]), causal=meta["causal"], pool=meta["pool"])
    model.load_state_dict(ck["state"]); return model.to(device).eval(), meta


def write_prediction(pred, docs, labels, outdir, name="prediction.json"):
    """pred: {doc_id: [label names]} ; validates ids / lengths / labels then writes prediction.json + submission.zip."""
    validate_prediction(pred, docs, labels)
    pj = os.path.join(outdir, name)
    with open(pj, "w", encoding="utf-8") as f: json.dump(pred, f)
    with zipfile.ZipFile(os.path.join(outdir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    return pj


def validate_prediction(pred, docs, labels):
    ok = set(labels)
    assert set(pred) == {d["id"] for d in docs}, "prediction ids != document ids"
    for d in docs:
        p = pred[d["id"]]
        assert isinstance(p, list) and len(p) == d["n"], f"{d['id']}: {len(p)} labels for {d['n']} sentences"
        assert all(x in ok for x in p), f"{d['id']}: unknown label"


def decode(lps, docs, labels):
    return {d["id"]: [labels[i] for i in lp.argmax(1)] for d, lp in zip(docs, lps)}


def save_lp(path_prefix, docs, lps):
    np.savez_compressed(path_prefix + "_lp.npz", lp=np.concatenate(lps, 0), lens=np.array([len(x) for x in lps]))
    with open(path_prefix + "_ids.json", "w") as f: json.dump([d["id"] for d in docs], f)


def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep or k.endswith(".mask")}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True); ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--init", required=True, help="Phase-1 weights (model_step45000.pt / model.pt / checkpoint.pt)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--method", required=True, choices=["full", "lora", "lorma", "adalora", "ia3", "prompt", "prefix", "kd"])
    ap.add_argument("--attn", default="bidir", choices=["causal", "bidir"])
    ap.add_argument("--pool", default="auto", choices=["auto", "mean", "last", "first"], help="auto: mean for bidir, last for causal")
    ap.add_argument("--epochs", type=float, default=6.0); ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=None); ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.01); ap.add_argument("--warmup", type=int, default=100)
    ap.add_argument("--window", type=int, default=512, help="tokens per window incl. prompt tokens (<= Phase-1 context)")
    ap.add_argument("--max_sent", type=int, default=128, help="max tokens per sentence (head 2/3 + tail 1/3 kept)")
    ap.add_argument("--marker", type=int, default=1, help="prepend <s> as a sentence-start marker")
    ap.add_argument("--train_core_frac", type=float, default=1.0); ap.add_argument("--infer_core_frac", type=float, default=0.5)
    ap.add_argument("--cw_pow", type=float, default=0.5, help="class weight = (1/freq)^cw_pow, 0 = unweighted")
    ap.add_argument("--cw_max", type=float, default=10.0)
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--targets", default="q_proj,k_proj,v_proj,out_proj,gate,up,down")
    ap.add_argument("--ada_init_r", type=int, default=24); ap.add_argument("--ada_target_r", type=int, default=16)
    ap.add_argument("--alpha_ada", type=float, default=48); ap.add_argument("--ada_orth", type=float, default=0.1)
    ap.add_argument("--ada_tinit_frac", type=float, default=0.1); ap.add_argument("--ada_tfinal_frac", type=float, default=0.3)
    ap.add_argument("--ia3_targets", default="k_proj,v_proj,down")
    ap.add_argument("--n_prompt", type=int, default=32); ap.add_argument("--n_prefix", type=int, default=32)
    ap.add_argument("--prefix_hidden", type=int, default=512, help="prefix reparametrisation MLP width (0 = free parameters)")
    ap.add_argument("--teacher", default=None); ap.add_argument("--student_layers", type=int, default=6)
    ap.add_argument("--kd_T", type=float, default=2.0); ap.add_argument("--kd_alpha", type=float, default=0.5)
    ap.add_argument("--full_data", action="store_true", help="also train on the dev docs; no model selection (use last epoch)")
    ap.add_argument("--time_budget_min", type=float, default=60.0)
    ap.add_argument("--eval_per_epoch", type=int, default=1)
    ap.add_argument("--max_train_docs", type=int, default=0); ap.add_argument("--max_dev_docs", type=int, default=0)
    ap.add_argument("--no_test", action="store_true")
    ap.add_argument("--n_folds", type=int, default=0, help="K-fold CV over ALL labelled docs (train+dev): train on k-1 folds, the held-out fold plays the role of 'dev'")
    ap.add_argument("--fold", type=int, default=0, help="which fold is held out (0..n_folds-1)"); ap.add_argument("--fold_seed", type=int, default=0, help="fold assignment seed (keep it identical for all runs you want to ensemble)")
    ap.add_argument("--select", default="auto", choices=["auto", "best", "last"], help="auto: best-on-dev epoch normally, LAST epoch in CV / full_data mode (so out-of-fold scores are not selected on)")
    ap.add_argument("--tta", type=int, default=1, help="final dev/test inference averaged over this many shifted window partitions (1 = off, max %d)" % len(TTA_OFFSETS))
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--cache", default=None)
    args = ap.parse_args()

    t_global = time.time()
    os.makedirs(args.out, exist_ok=True)
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    default_lr = dict(full=5e-5, kd=1e-4, lora=1e-3, lorma=1e-3, adalora=1e-3, ia3=1e-2, prompt=2e-2, prefix=5e-3)
    base_lr = args.lr if args.lr is not None else default_lr[args.method]
    print(f"== method={args.method} attn={args.attn} lr={base_lr} lr_head={args.lr_head} device={device}")

    
    tok = ByteLevelBPETokenizer.load(args.tokenizer)
    labels, label2idx = load_vocab(load_json(os.path.join(args.data_dir, "label_vocab.json")))
    K = len(labels)
    enc = WordBPE(tok)
    D_ = get_docs(args, enc, labels, label2idx, args.tokenizer)
    train_docs, dev_docs, test_docs = D_["train"], D_["dev"], D_["test"]
    if args.n_folds > 1:
        assert 0 <= args.fold < args.n_folds and not args.full_data, "--fold must be in [0, n_folds) and cannot be combined with --full_data"
        pool_docs = bal_docs(train_docs + dev_docs, args.max_train_docs)
        fold_of = make_folds(pool_docs, args.n_folds, args.fold_seed)
        train_docs = [d for d in pool_docs if fold_of[d["id"]] != args.fold]; dev_docs = [d for d in pool_docs if fold_of[d["id"]] == args.fold]
        print(f"  CV fold {args.fold}/{args.n_folds} (fold_seed={args.fold_seed}): train={len(train_docs)} held-out={len(dev_docs)} docs")
    else:
        train_docs = bal_docs(train_docs, args.max_train_docs); dev_docs = bal_docs(dev_docs, args.max_dev_docs)
        if args.full_data: train_docs = train_docs + dev_docs
    use_last = args.full_data or args.select == "last" or (args.select == "auto" and args.n_folds > 1)

    
    teacher = None
    if args.method == "kd":
        assert args.teacher, "--teacher runs/<full_run>/best.pt is required for kd"
        teacher, tmeta = load_teacher(args.teacher, device)
        layer_map = [int(round(x)) for x in np.linspace(0, tmeta["lm_cfg"]["n_layers"] - 1, args.student_layers)]
        print(f"  KD student = teacher blocks {layer_map}; teacher params={count_params(teacher)[0] / 1e6:.1f}M")
        lm, lm_cfg = load_phase1(args.init, layer_map=layer_map)
        args.attn = "causal" if tmeta["causal"] else "bidir"      
        args.pool, args.marker, args.max_sent = tmeta["pool"], tmeta["marker"], tmeta["max_sent"]
    else:
        lm, lm_cfg = load_phase1(args.init)
    pool = args.pool if args.pool != "auto" else ("last" if args.attn == "causal" else "mean")
    n_virtual = args.n_prompt if args.method == "prompt" else 0
    budget = args.window - n_virtual
    globals()["PAD_CAP"] = budget
    assert args.window <= lm_cfg["max_seq_len"], "window longer than the Phase-1 context (RoPE tables are built for 512)"
    assert args.max_sent + 1 <= budget, "max_sent must fit into one window"
    finalize_docs(train_docs, args.max_sent, args.marker); finalize_docs(dev_docs, args.max_sent, args.marker); finalize_docs(test_docs, args.max_sent, args.marker)
    cnt = np.zeros(K)
    for d in train_docs: cnt += np.bincount(d["y"], minlength=K)
    print(f"  docs: train={len(train_docs)} (CL={sum(d['dom'] == 0 for d in train_docs)}, IT={sum(d['dom'] == 1 for d in train_docs)}) dev={len(dev_docs)} test={len(test_docs)}"
          f" | sentences train={int(cnt.sum())} dev={sum(d['n'] for d in dev_docs)} test={sum(d['n'] for d in test_docs)}")
    print("  train label counts:", {labels[i]: int(c) for i, c in enumerate(cnt)})

    model = RRModel(lm, K, causal=(args.attn == "causal"), pool=pool)
    if args.method == "kd":
        with torch.no_grad(): model.head.load_state_dict(teacher.head.state_dict()); model.pos_emb.load_state_dict(teacher.pos_emb.state_dict()); model.dom_emb.load_state_dict(teacher.dom_emb.state_dict())
    before = {id(p) for p in model.parameters()}
    ctl = apply_method(model, args)
    mark_new_params_trainable(model, before)
    model.to(device)
    tot, trn = count_params(model)
    print(f"  params total={tot / 1e6:.2f}M trainable={trn / 1e6:.3f}M ({100 * trn / tot:.2f}%) | pool={pool} budget={budget} tokens/window")

    cw = np.ones(K)
    if args.cw_pow > 0:
        fr = np.maximum(cnt, 1) / cnt.sum(); cw = np.minimum((1.0 / fr) ** args.cw_pow, 1e9); cw = cw / (cw * fr).sum(); cw = np.minimum(cw, args.cw_max)
    cw_t = torch.tensor(cw, dtype=torch.float, device=device)
    print("  class weights:", {labels[i]: round(float(w), 2) for i, w in enumerate(cw)})

    decay, no_decay, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (head_p if is_head_param(n) else decay if (p.ndim >= 2 and not n.startswith(("prompt", "prefix"))) else no_decay).append(p)
    groups = [dict(params=head_p, lr=args.lr_head, weight_decay=0.0), dict(params=no_decay, lr=base_lr, weight_decay=0.0),
              dict(params=decay, lr=base_lr, weight_decay=args.wd)]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    rng = random.Random(args.seed)
    meta = meta_of(args, lm_cfg, labels, model)

    def epoch_items():
        items = []
        for di, d in enumerate(train_docs):
            for lo, hi, a, b in make_windows(d["lens"], budget, args.train_core_frac, first_frac=rng.uniform(0.15, 1.0)):
                items.append((di, lo, hi, a, b))
        rng.shuffle(items); return items

    n_items = len(epoch_items())
    steps_per_epoch = max(1, n_items // args.bs)
    planned = int(args.epochs * steps_per_epoch)
    total, step, best_f1, best_step, t_bench = planned, 0, -1.0, 0, None
    eval_every = max(1, steps_per_epoch // max(1, args.eval_per_epoch))
    best_state = None
    print(f"  windows/epoch={n_items} steps/epoch={steps_per_epoch} planned steps={planned}")
    if args.method == "adalora":
        ctl = AdaLoRACtl(model, args.ada_init_r, args.ada_target_r, tinit=max(40, int(args.ada_tinit_frac * planned)),
                         tfinal=int(args.ada_tfinal_frac * planned), orth=args.ada_orth)
        print(f"  AdaLoRA: {len(ctl.mods)} modules, budget {ctl.init_budget} -> {ctl.target_budget} triplets, tinit={ctl.tinit} tfinal={ctl.tfinal}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()

    def run_eval(tag):
        nonlocal best_f1, best_step, best_state
        if args.full_data:                      
            best_state, best_step = trainable_state(model), step; return
        lps = infer_docs(model, dev_docs, budget, args.infer_core_frac, device, amp)
        sc = score_docs(dev_docs, lps)
        print(f"  [eval {tag}] step={step} dev macro-F1={sc['F1']:.4f} P={sc['P']:.4f} R={sc['R']:.4f}", flush=True)
        if use_last:                            
            best_f1, best_step, best_state = sc["F1"], step, trainable_state(model)
        elif sc["F1"] > best_f1:
            best_f1, best_step, best_state = sc["F1"], step, trainable_state(model)
        model.train()

    model.train()
    t_start = time.time(); done = False; ep = 0; loss_ema = None
    while not done:
        order_w = epoch_items(); ep += 1
        for bi in range(0, len(order_w) - args.bs + 1, args.bs):
            if step == 5: t_bench = time.time()
            if step == 35 and t_bench is not None:                      
                sps = (time.time() - t_bench) / 30
                left = args.time_budget_min * 60 * 0.80 - (time.time() - t_global)
                total = max(60, min(planned, 35 + int(left / sps)))
                print(f"  speed={1 / sps:.2f} steps/s -> total steps={total} (planned {planned})", flush=True)
            if step >= total or (time.time() - t_global) > args.time_budget_min * 60 * 0.95: done = True; break
            lr_scale = lr_at(step, total, args.warmup, 1.0)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_scale
            with torch.autocast("cuda", enabled=amp):
                logits, lab = batch_forward(model, train_docs, order_w[bi:bi + args.bs], device)
            m = lab != IGN
            lf = logits.float()[m]; y = lab[m]
            ce = F.cross_entropy(lf, y, weight=cw_t)
            loss = ce
            if teacher is not None:
                ids_ = None
                with torch.no_grad(), torch.autocast("cuda", enabled=amp):
                    tl, _ = batch_forward(teacher, train_docs, order_w[bi:bi + args.bs], device)
                tl = tl.float()[m]; T = args.kd_T
                kd = F.kl_div(F.log_softmax(lf / T, -1), F.softmax(tl / T, -1), reduction="batchmean") * T * T
                loss = args.kd_alpha * ce + (1 - args.kd_alpha) * kd
            if ctl is not None: loss = loss + ctl.orth_loss()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            if ctl is not None: ctl.step(step + 1, total)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1
            loss_ema = loss.item() if loss_ema is None else 0.98 * loss_ema + 0.02 * loss.item()
            if step % 50 == 0:
                print(f"  step {step}/{total} ep {ep} loss {loss_ema:.4f} lr {base_lr * lr_scale:.2e} {(time.time() - t_start) / 60:.1f}min", flush=True)
            if step % eval_every == 0 and step < total and step >= 35: run_eval("mid")
    train_min = (time.time() - t_start) / 60
    run_eval("final")
    peak_gb = torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else 0.0

    
    model.load_state_dict(best_state, strict=False)
    if args.method == "full":
        torch.save({"state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "meta": meta}, os.path.join(args.out, "best.pt"))
    elif best_state is not None:
        torch.save(best_state, os.path.join(args.out, "trainable.pt"))
    t0 = time.time()
    dev_lps = infer_docs(model, dev_docs, budget, args.infer_core_frac, device, amp)
    n_tok = sum(sum(d["lens"]) for d in dev_docs); inf_tps = n_tok / max(1e-6, time.time() - t0)
    sc_plain = score_docs(dev_docs, dev_lps)
    if args.tta > 1:
        dev_lps = infer_docs(model, dev_docs, budget, args.infer_core_frac, device, amp, n_tta=args.tta)
        print(f"  TTA x{args.tta}: held-out macro-F1 {sc_plain['F1']:.4f} (single pass) -> {score_docs(dev_docs, dev_lps)['F1']:.4f}")
    save_lp(os.path.join(args.out, "dev"), dev_docs, dev_lps)
    sc = score_docs(dev_docs, dev_lps)
    per_dom = {}
    for dn, dm in SPLITS:
        sel = [i for i, d in enumerate(dev_docs) if d["dom"] == dm]
        if sel: per_dom[dn] = score_docs([dev_docs[i] for i in sel], [dev_lps[i] for i in sel])["F1"]
    print(f"  dev macro-F1={sc['F1']:.4f} P={sc['P']:.4f} R={sc['R']:.4f} | per-domain F1: {per_dom}")
    for i, l in enumerate(labels):
        print(f"    {l:24s} P={sc['per_p'][i]:.3f} R={sc['per_r'][i]:.3f} F1={sc['per_f1'][i]:.3f} support={sc['support'][i]}")
    with open(os.path.join(args.out, "dev_pred.json"), "w") as f: json.dump(decode(dev_lps, dev_docs, labels), f)

    res = dict(method=args.method, attn=args.attn, pool=pool, params_total=tot, params_trainable=trn, train_minutes=train_min, steps=step,
               planned_steps=planned, peak_gpu_gb=peak_gb, infer_tokens_per_s=inf_tps, best_step=best_step, full_data=args.full_data,
               dev_macro_F1=sc["F1"], dev_macro_P=sc["P"], dev_macro_R=sc["R"], dev_per_domain_F1=per_dom,
               dev_macro_F1_single_pass=sc_plain["F1"], tta=args.tta, n_folds=args.n_folds, fold=args.fold if args.n_folds > 1 else None, select_last=use_last,
               per_label={l: dict(P=sc["per_p"][i], R=sc["per_r"][i], F1=sc["per_f1"][i], support=sc["support"][i]) for i, l in enumerate(labels)},
               args=vars(args), n_dev_docs=len(dev_docs))
    if ctl is not None and args.method == "adalora": res["adalora_rank"] = ctl.rank_report(); print("  AdaLoRA final ranks:", res["adalora_rank"])
    if not args.no_test:
        test_lps = infer_docs(model, test_docs, budget, args.infer_core_frac, device, amp, n_tta=args.tta)
        save_lp(os.path.join(args.out, "test"), test_docs, test_lps)
        pred = decode(test_lps, test_docs, labels)
        write_prediction(pred, test_docs, labels, args.out)
        res["n_test_docs"], res["n_test_sentences"] = len(pred), sum(len(v) for v in pred.values())
        print(f"  wrote {args.out}/submission.zip  ({res['n_test_sentences']} sentences in {len(pred)} docs)")
    with open(os.path.join(args.out, "results.json"), "w") as f: json.dump(res, f, indent=1, default=str)
    print(f"== done in {(time.time() - t_global) / 60:.1f} min | dev macro-F1 = {res['dev_macro_F1']:.4f}")


if __name__ == "__main__":
    main()

Writing rr.py


In [2]:
%%writefile rr_ed.py

"""
rr_ed.py -- CS781 Phase 3 (Rhetorical Roles) on top of YOUR Phase-1 *encoder-decoder* model (RoPE + RMSNorm + GeGLU, 6+6 layers, d640, 96M params).

Thin layer over rr.py (keep both files in the same folder): swaps in the enc-dec trunk + its adaptation wiring and then calls rr.main(), so windows,
sentence pooling, class weights, KD, K-fold, TTA, log-prob dumps, ensemble3.py / ensemble_cv.py behave EXACTLY as for the decoder.

Two ways of using the model (--ed_mode):
  enc  (default)  the window goes through the bidirectional ENCODER only; sentence vectors = mean over the sentence's encoder states.
                  The decoder half (6 layers + cross-attention, ~49M params) is NOT built, NOT trained.  Say so in your report.
  both            encoder AND decoder: the decoder is teacher-forced with the window itself ([BOS] + tokens shifted right), causal self-attention + cross-attention
                  to the encoder memory; sentence vector = [mean(encoder states) ++ mean(decoder states)].  Window <= 400 (the decoder was pre-trained on 400 positions).
Methods:  full | lora | lorma | adalora | ia3 | prompt | kd      (prompt tuning prepends virtual tokens to the encoder input; no prefix tuning here)
  python rr_ed.py --data_dir DATA --tokenizer bpe_tokenizer.json --init model.pt --out runs/ed_lorma --method lorma
KD: the student keeps every 2nd layer (encoder 6 -> 3, and decoder 6 -> 3 in `both` mode):  --student_layers 3  (added automatically).

Sections: A. enc-dec model (copied from your notebook; ONLY change: optional key-padding masks)  B. loader  C. wrapper + PEFT wiring  D. entry point
"""
import math
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import rr

ED_MODE = "enc"          





class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos[:, :, offset:offset + T, :].to(x.dtype)
        sin = self.sin[:, :, offset:offset + T, :].to(x.dtype)
        return x * cos + rotate_half(x) * sin


class Attention(nn.Module):
    """encoder self-attention (bidirectional, RoPE) | decoder self-attention (causal, RoPE) | cross-attention (no RoPE)."""
    def __init__(self, d_model, n_heads, dropout=0.0, rope=None, causal=False):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope, self.causal, self.dropout = rope, causal, dropout
        self.resid_dropout = nn.Dropout(dropout)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x, context=None, cache=None, offset=0, key_mask=None):       
        B, T, C = x.shape
        q = self._split(self.q_proj(x))
        attn_mask = None
        if context is not None:                       
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k_proj(context)), self._split(self.v_proj(context))
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
            if key_mask is not None: attn_mask = key_mask[:, None, None, :]
        else:                                         
            k, v = self._split(self.k_proj(x)), self._split(self.v_proj(x))
            if self.rope is not None:
                q, k = self.rope(q, offset), self.rope(k, offset)
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2)
                    v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            is_causal = self.causal and cache is None
            if key_mask is not None:                  
                attn_mask = key_mask[:, None, None, :]
                if is_causal:
                    attn_mask = attn_mask & torch.ones(T, T, dtype=torch.bool, device=x.device).tril()[None, None]
                    is_causal = False
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0, is_causal=is_causal)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class GeGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down(F.gelu(self.gate(x)) * self.up(x)))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, rope, dropout):
        super().__init__()
        self.norm_attn = RMSNorm(d_model)
        self.attn = Attention(d_model, n_heads, dropout, rope=rope, causal=False)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = GeGLU(d_model, d_ff, dropout)

    def forward(self, x, key_mask=None):
        x = x + self.attn(self.norm_attn(x), key_mask=key_mask)
        return x + self.ffn(self.norm_ffn(x))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, rope, dropout):
        super().__init__()
        self.norm_self = RMSNorm(d_model)
        self.self_attn = Attention(d_model, n_heads, dropout, rope=rope, causal=True)
        self.norm_cross = RMSNorm(d_model)
        self.cross_attn = Attention(d_model, n_heads, dropout, rope=None, causal=False)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = GeGLU(d_model, d_ff, dropout)

    def forward(self, x, memory, mem_mask=None):
        x = x + self.self_attn(self.norm_self(x))                 
        x = x + self.cross_attn(self.norm_cross(x), context=memory, key_mask=mem_mask)
        return x + self.ffn(self.norm_ffn(x))


class EDLM(nn.Module):
    """Container with the names rr.RRModel expects (token_emb / blocks / final_norm) = the ENCODER; dec_blocks / dec_norm exist only in mode `both`."""
    def __init__(self, vocab_size, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff, enc_len, dec_len, dropout, rope_base, mode):
        super().__init__()
        rope_len = max(enc_len, dec_len) + 8
        d_head = d_model // n_heads
        self.enc_rope = RotaryEmbedding(d_head, rope_len, rope_base)
        self.dec_rope = RotaryEmbedding(d_head, rope_len, rope_base)
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, self.enc_rope, dropout) for _ in range(n_enc_layers)])
        self.final_norm = RMSNorm(d_model)
        self.dec_blocks, self.dec_norm = None, None
        if mode == "both":
            self.dec_blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, self.dec_rope, dropout) for _ in range(n_dec_layers)])
            self.dec_norm = RMSNorm(d_model)





DEFAULT_ED_CFG = dict(vocab_size=12000, d_model=640, n_enc_layers=6, n_dec_layers=6, n_heads=10, d_ff=2560,
                      enc_len=512, dec_len=400, dropout=0.1, rope_base=10000.0)


def build_lm_ed(cfg):
    c = {**DEFAULT_ED_CFG, **{k: v for k, v in cfg.items() if k in DEFAULT_ED_CFG}}
    mode = cfg.get("mode", ED_MODE)
    lm = EDLM(**c, mode=mode)
    c["mode"], c["n_layers"], c["max_seq_len"] = mode, c["n_enc_layers"], c["enc_len"]       
    return lm, c


def load_phase1_ed(path, n_layers=None, layer_map=None):
    """Accepts model.pt ({state_dict, config}), checkpoint.pt ({model_state, run_config}) or a raw state_dict.  Keeps only the layers in `layer_map` (KD student)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = {}
    if isinstance(ck, dict) and "state_dict" in ck:
        sd, cfg = ck["state_dict"], dict(ck.get("config", {}))
    elif isinstance(ck, dict) and "model_state" in ck:
        sd = ck["model_state"]; rc = ck.get("run_config", {}) or {}
        cfg = {k: v for k, v in dict(vocab_size=rc.get("VOCAB_SIZE"), d_model=rc.get("D_MODEL"), n_enc_layers=rc.get("N_ENC_LAYERS"), n_dec_layers=rc.get("N_DEC_LAYERS"),
                                     n_heads=rc.get("N_HEADS"), d_ff=rc.get("D_FF"), enc_len=rc.get("ENC_LEN"), dec_len=rc.get("DEC_LEN"), rope_base=rc.get("ROPE_BASE")).items() if v is not None}
    else:
        sd = ck
    sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
    cfg["n_enc_layers"] = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("enc_blocks."))
    cfg["n_dec_layers"] = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("dec_blocks."))
    cfg["vocab_size"], cfg["d_model"] = sd["token_emb.weight"].shape
    cfg["d_ff"] = sd["enc_blocks.0.ffn.gate.weight"].shape[0]
    mode = ED_MODE
    
    new = {}
    for k, v in sd.items():
        if k.startswith("enc_blocks."): new["blocks." + k[len("enc_blocks."):]] = v
        elif k.startswith("enc_norm."): new["final_norm." + k[len("enc_norm."):]] = v
        elif k.startswith("dec_blocks.") or k.startswith("dec_norm.") or k.startswith("token_emb."):
            if mode == "both" or k.startswith("token_emb."): new[k] = v
    sd = new
    if layer_map is not None:                                                 
        dmap = [int(round(x)) for x in torch.linspace(0, cfg["n_dec_layers"] - 1, len(layer_map)).tolist()]
        out = {k: v for k, v in sd.items() if not k.startswith(("blocks.", "dec_blocks."))}
        for pref, mp in (("blocks.", layer_map), ("dec_blocks.", dmap)):
            for ni, oi in enumerate(mp):
                for k, v in sd.items():
                    if k.startswith(f"{pref}{oi}."): out[f"{pref}{ni}." + k.split(".", 2)[2]] = v
        sd, cfg["n_enc_layers"], cfg["n_dec_layers"] = out, len(layer_map), len(layer_map)
    cfg["mode"] = mode
    lm, cfg = build_lm_ed(cfg)
    missing, unexpected = lm.load_state_dict(sd, strict=False)
    bad = list(missing) + list(unexpected)
    assert not bad, f"checkpoint/architecture mismatch: {bad[:6]}"
    return lm, cfg





class EDRR(rr.RRModel):
    """Enc-dec trunk + the same sentence pooling / head as the decoder."""
    def __init__(self, lm, K, causal=False, pool="mean", head_dropout=0.1, pos_dim=32, dom_dim=8):
        super().__init__(lm, K, causal=causal, pool=pool, head_dropout=head_dropout, pos_dim=pos_dim, dom_dim=dom_dim)
        self.emb_dropout = lm.emb_dropout
        self.dec_blocks, self.dec_norm = lm.dec_blocks, lm.dec_norm
        if self.dec_blocks is not None:                                       
            D = self.token_emb.embedding_dim
            self.head = nn.Linear(2 * D + pos_dim + dom_dim, K)

    def trunk(self, ids, pad_mask):
        B, L = ids.shape
        x = self.emb_dropout(self.token_emb(ids)); P = 0; km = pad_mask
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(x.dtype), x], 1)
            if pad_mask is not None: km = torch.cat([torch.ones(B, P, dtype=torch.bool, device=ids.device), pad_mask], 1)
        for b in self.blocks: x = b(x, km)
        mem = self.final_norm(x)
        out = mem[:, P:]
        if self.dec_blocks is not None:
            dec_in = torch.cat([torch.full((B, 1), rr.BOS_ID, dtype=ids.dtype, device=ids.device), ids[:, :-1]], 1)
            y = self.emb_dropout(self.token_emb(dec_in))
            for b in self.dec_blocks: y = b(y, mem, km)
            out = torch.cat([out, self.dec_norm(y)], -1)
        return out


ATTN_T = ("q_proj", "k_proj", "v_proj", "out_proj")
_DEFAULT_TARGETS = "q_proj,k_proj,v_proj,out_proj,gate,up,down"


def _parents(model, t):
    """modules that own linear `t`, over the encoder and (mode both) the decoder stack"""
    ps = []
    for b in model.blocks: ps.append(b.attn if t in ATTN_T else b.ffn)
    if model.dec_blocks is not None:
        for b in model.dec_blocks: ps += [b.self_attn, b.cross_attn] if t in ATTN_T else [b.ffn]
    return ps


def apply_method_ed(model, args):
    m = args.method
    for p in model.parameters(): p.requires_grad = False
    for n, p in model.named_parameters():
        if rr.is_head_param(n): p.requires_grad = True
    tg = [t.strip() for t in args.targets.split(",") if t.strip()]

    def wrap_linears(factory, names):
        for t in names:
            for par in _parents(model, t): setattr(par, t, factory(getattr(par, t)))

    ctl = None
    if m in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    elif m == "lora":
        wrap_linears(lambda l: rr.LoRALinear(l, args.rank, args.alpha, args.lora_dropout), tg)
    elif m == "lorma":
        wrap_linears(lambda l: rr.LoRMAPlusLinear(l, args.rank, args.alpha), tg)
    elif m == "adalora":
        wrap_linears(lambda l: rr.AdaLoRALinear(l, args.ada_init_r, args.alpha_ada, args.lora_dropout), tg)
        ctl = "pending"
    elif m == "ia3":
        for t in [x.strip() for x in args.ia3_targets.split(",") if x.strip()]:
            for par in _parents(model, t):
                lin = getattr(par, t); setattr(par, t, rr.IA3In(lin) if t == "down" else rr.IA3Out(lin))
    elif m == "prompt":
        ids = torch.randint(260, model.token_emb.num_embeddings, (args.n_prompt,))
        model.prompt = nn.Parameter(model.token_emb.weight[ids].detach().clone())
    else:
        raise ValueError(f"enc-dec pipeline supports full | lora | lorma | adalora | ia3 | prompt | kd, got {m}")
    return ctl


def _load_teacher(path, device):
    """rr.load_teacher + make the student use the teacher's mode (enc / both)."""
    global ED_MODE
    ck = torch.load(path, map_location="cpu", weights_only=False)
    ED_MODE = ck["meta"]["lm_cfg"].get("mode", ED_MODE)
    return _orig_load_teacher(path, device)


_orig_load_teacher = rr.load_teacher


def _patched():
    rr.load_phase1, rr.build_lm = load_phase1_ed, build_lm_ed
    rr.RRModel, rr.apply_method, rr.load_teacher = EDRR, apply_method_ed, _load_teacher





if __name__ == "__main__":
    argv = sys.argv
    if "--ed_mode" in argv:
        i = argv.index("--ed_mode"); ED_MODE = argv[i + 1]; del argv[i:i + 2]
    assert ED_MODE in ("enc", "both"), "--ed_mode enc|both"
    if "--window" not in argv: argv += ["--window", "512" if ED_MODE == "enc" else "400"]
    if ED_MODE == "both" and int(argv[argv.index("--window") + 1]) > 400:
        print("WARNING: the decoder was pre-trained on 400 positions; --window > 400 in `both` mode extrapolates its RoPE")
    if "--method" in argv and argv[argv.index("--method") + 1] == "kd" and "--student_layers" not in argv: argv += ["--student_layers", "3"]
    sys.argv = argv
    _patched()
    rr.main()

Writing rr_ed.py


In [3]:
%%writefile ensemble_cv.py

"""
ensemble_cv.py -- combine K-FOLD runs (rr.py --n_folds K --fold f) into one honest evaluation + one test submission.

A *member* is a run-name prefix; its folds are the run dirs  <prefix>_f0 ... <prefix>_f{K-1}  (each holds dev_lp.npz = log-probs of the
HELD-OUT fold, test_lp.npz = log-probs of the test set).   Example:   cv_lora_s1  ->  cv_lora_s1_f0 .. cv_lora_s1_f4

  python ensemble_cv.py --runs_dir runs --data_dir data --members cv_lora_s1,cv_lorma_s1,cv_adalora_s1 --viterbi --out subs/cv_ens

What it does
  1. OOF (out-of-fold) predictions: every labelled document (train+dev, ~90) is predicted by a model that never saw it -> ONE macro-F1 over
     all ~20k labelled sentences instead of the 10-document dev set (which is far too small to rank anything; rare roles are often absent from it).
  2. Test predictions = probability average over the K fold models (a K-model ensemble for free).
  3. Members are averaged (prob or log-prob mean); --greedy picks members by OOF macro-F1 (optimistic: members chosen on the data they are scored on).
  4. --viterbi : sequence decoding.  emission = log p(y|x) - beta*log prior(y);   score = sum emission + lam * log T(y_{t-1}, y_t);   T from the TRAINING
     labels (per-domain, mixed with the global matrix).  (lam, beta) are picked on OOF; for OOF scoring T/prior are re-estimated WITHOUT the fold being scored.
  5. --bias_tune : per-class additive bias on OOF (coordinate ascent).
  Every tuned step also prints a NESTED estimate: its settings are re-tuned on the other folds and scored on the held-out fold, which is the
  number to quote (the in-sample OOF number is optimistic).  A step is only applied if its nested gain is positive (use --force to override).
  --boot N : bootstrap over documents -> 95% interval for the final OOF macro-F1 (shows how little a +0.005 difference means).
"""
import argparse, glob, itertools, json, os, sys, zipfile
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rr

LAMS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]
BETAS = [0.0, 0.5, 1.0]



def load_gold(data_dir, labels, label2idx):
    """id -> dict(y, dom, n) for every labelled document (CL/IT train+dev)."""
    gold = {}
    for kind in ("train", "dev"):
        for dn, dom in rr.SPLITS:
            for did, sents, labs in rr.iter_raw_docs(rr.load_json(os.path.join(data_dir, f"{dn}_{kind}.json"))):
                assert did not in gold, f"duplicate id {did}"
                gold[did] = dict(y=np.array([rr.label_to_idx(l, label2idx, labels) for l in labs]), dom=dom, n=len(sents))
    return gold


def split_lp(z, ids):
    lens = z["lens"]; lp = z["lp"]; off = np.concatenate([[0], np.cumsum(lens)])
    return {i: lp[off[k]:off[k + 1]].astype(np.float64) for k, i in enumerate(ids)}


def load_member(runs_dir, prefix, gold):
    """-> dict(oof={id: lp}, fold_of={id: f}, test={id: lp averaged over folds (probabilities)}, test_ids=[...]) or None."""
    dirs = sorted(glob.glob(os.path.join(runs_dir, f"{prefix}_f[0-9]*")), key=lambda p: int(p.rsplit("_f", 1)[1]))
    dirs = [d for d in dirs if os.path.exists(os.path.join(d, "dev_lp.npz"))]
    if not dirs: return None
    oof, fold_of, test_acc, test_ids = {}, {}, None, None
    for d in dirs:
        f = int(d.rsplit("_f", 1)[1])
        ids = json.load(open(os.path.join(d, "dev_ids.json")))
        for i, lp in split_lp(np.load(os.path.join(d, "dev_lp.npz")), ids).items():
            assert i in gold and i not in oof, f"{prefix}: doc {i} appears twice / is unknown (mixed fold seeds?)"
            assert len(lp) == gold[i]["n"], f"{prefix}: {i} has {len(lp)} rows for {gold[i]['n']} sentences"
            oof[i], fold_of[i] = lp, f
        tp = os.path.join(d, "test_lp.npz")
        if os.path.exists(tp):
            tids = json.load(open(os.path.join(d, "test_ids.json")))
            t = split_lp(np.load(tp), tids)
            if test_ids is None: test_ids, test_acc = tids, {i: 0.0 for i in tids}
            assert tids == test_ids, f"{prefix}: test doc order differs between folds"
            for i in tids: test_acc[i] = test_acc[i] + np.exp(t[i])
    missing = set(gold) - set(oof)
    if missing:
        print(f"  !! {prefix}: only {len(oof)}/{len(gold)} labelled docs covered ({len(dirs)} fold runs found) -> member skipped (finish or delete it)")
        return None
    nf = len(set(fold_of.values()))
    test = {i: np.log(np.maximum(test_acc[i] / len(dirs), 1e-12)) for i in test_ids} if test_ids and all(os.path.exists(os.path.join(d, "test_lp.npz")) for d in dirs) else None
    return dict(oof=oof, fold_of=fold_of, test=test, test_ids=test_ids, n_folds=nf)



def prf(y, p, K=13): return rr.macro_prf(y, p, K)


def combine(lps, mode="prob"):
    """lps: list of (n, K) log-prob arrays (equal weights; greedy repeats = weights)."""
    if mode == "prob": return np.log(np.maximum(sum(np.exp(l) for l in lps) / len(lps), 1e-12))
    return sum(lps) / len(lps)


def fit_transitions(docs, K, alpha=0.5, mix=0.7):
    """docs: list of dict(y, dom).  -> logprior (K,), logT[dom] (K,K), logstart[dom] (K,).  Per-domain matrices are mixed with the global one."""
    def mats(ds):
        c = np.full((K, K), alpha); s = np.full(K, alpha); pr = np.full(K, alpha)
        for d in ds:
            y = d["y"]; s[y[0]] += 1; np.add.at(c, (y[:-1], y[1:]), 1); np.add.at(pr, y, 1)
        return c / c.sum(1, keepdims=True), s / s.sum(), pr / pr.sum()
    Tg, sg, pg = mats(docs); T, S = {}, {}
    for dom in (0, 1):
        ds = [d for d in docs if d["dom"] == dom]
        if ds:
            Td, sd, _ = mats(ds); T[dom] = np.log(mix * Td + (1 - mix) * Tg); S[dom] = np.log(mix * sd + (1 - mix) * sg)
        else: T[dom], S[dom] = np.log(Tg), np.log(sg)
    return np.log(pg), T, S


def viterbi(em, logT, logS, lam):
    """em (n,K) emission scores; returns best label path under  sum em_t + lam*(logS[y_0] + sum logT[y_{t-1},y_t])."""
    n, K = em.shape
    if lam == 0 or n == 1: return em.argmax(1)
    delta = em[0] + lam * logS; back = np.zeros((n, K), np.int16); LT = lam * logT
    for t in range(1, n):
        cand = delta[:, None] + LT; back[t] = cand.argmax(0); delta = cand.max(0) + em[t]
    path = np.zeros(n, np.int64); path[-1] = delta.argmax()
    for t in range(n - 1, 0, -1): path[t - 1] = back[t, path[t]]
    return path


def decode_all(ids, lp, dom_of, trans, lam, beta, bias=None):
    """trans: callable(id) -> (logprior, logT, logS) for this doc.  Returns {id: path}."""
    out = {}
    for i in ids:
        logpr, T, S = trans(i); e = lp[i] - beta * logpr + (0 if bias is None else bias)
        out[i] = viterbi(e, T[dom_of[i]], S[dom_of[i]], lam)
    return out


def tune_bias(y, lp, K, grid=np.linspace(-1, 1, 9), rounds=2):
    b = np.zeros(K); best = prf(y, (lp + b).argmax(1), K)["F1"]
    for _ in range(rounds):
        for c in range(K):
            cand = []
            for g in grid:
                bb = b.copy(); bb[c] = g; cand.append((prf(y, (lp + bb).argmax(1), K)["F1"], -abs(g), g))
            f, _, g = max(cand)
            if f > best + 1e-9: best, b[c] = f, g
    return b, best



def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs_dir", required=True); ap.add_argument("--data_dir", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--members", default="", help="comma list of run-name prefixes (folds = <prefix>_f0..)")
    ap.add_argument("--all", action="store_true", help="every prefix that has fold runs in runs_dir (excluding smoke*/tmp*)")
    ap.add_argument("--mode", default="prob", choices=["prob", "logp"])
    ap.add_argument("--greedy", action="store_true"); ap.add_argument("--greedy_steps", type=int, default=10)
    ap.add_argument("--viterbi", action="store_true"); ap.add_argument("--bias_tune", action="store_true")
    ap.add_argument("--boot", type=int, default=200); ap.add_argument("--force", action="store_true", help="apply Viterbi/bias even if their nested gain is not positive")
    ap.add_argument("--no_test", action="store_true")
    args = ap.parse_args()

    labels, label2idx = rr.load_vocab(rr.load_json(os.path.join(args.data_dir, "label_vocab.json"))); K = len(labels)
    gold = load_gold(args.data_dir, labels, label2idx)
    ids_all = sorted(gold); dom_of = {i: gold[i]["dom"] for i in ids_all}
    print(f"{len(gold)} labelled docs (CL={sum(v['dom'] == 0 for v in gold.values())}, IT={sum(v['dom'] == 1 for v in gold.values())}), "
          f"{sum(v['n'] for v in gold.values())} sentences")

    if args.all:
        pre = {os.path.basename(os.path.dirname(p)).rsplit("_f", 1)[0] for p in glob.glob(os.path.join(args.runs_dir, "*_f[0-9]*", "dev_lp.npz"))}
        prefixes = sorted(p for p in pre if not p.startswith(("smoke", "tmp")))
    else: prefixes = [m for m in args.members.split(",") if m]
    assert prefixes, "give --members a,b,c or --all"
    mem = {}
    for p in prefixes:
        m = load_member(args.runs_dir, p, gold)
        if m is not None: mem[p] = m
    assert mem, "no complete member found"
    names = list(mem)
    fold_of = mem[names[0]]["fold_of"]
    for n in names: assert mem[n]["fold_of"] == fold_of, f"{n}: fold assignment differs from {names[0]} (different --fold_seed / --n_folds?)"
    folds = sorted(set(fold_of.values())); nf = len(folds)
    y_of = {i: gold[i]["y"] for i in ids_all}; Y = np.concatenate([y_of[i] for i in ids_all])
    fold_vec = np.concatenate([[fold_of[i]] * gold[i]["n"] for i in ids_all])
    doc_vec = np.concatenate([[k] * gold[i]["n"] for k, i in enumerate(ids_all)])
    dom_vec = np.concatenate([[dom_of[i]] * gold[i]["n"] for i in ids_all])
    print(f"{nf} folds, docs per fold: {[sum(1 for i in ids_all if fold_of[i] == f) for f in folds]}")

    def cat(lpd): return np.concatenate([lpd[i] for i in ids_all], 0)
    def fold_f1s(P): return [prf(Y[fold_vec == f], P[fold_vec == f], K)["F1"] for f in folds]

    print("\nsingle members (OOF = every labelled doc predicted by a model that never saw it):")
    print(f"  {'member':30s} {'OOF F1':>7s} {'P':>6s} {'R':>6s} {'CL':>6s} {'IT':>6s} {'fold-mean±sd':>14s}")
    for n in names:
        P = cat(mem[n]["oof"]).argmax(1); s = prf(Y, P, K); ff = fold_f1s(P)
        print(f"  {n:30s} {s['F1']:7.4f} {s['P']:6.3f} {s['R']:6.3f} {prf(Y[dom_vec == 0], P[dom_vec == 0], K)['F1']:6.3f} {prf(Y[dom_vec == 1], P[dom_vec == 1], K)['F1']:6.3f} {np.mean(ff):8.4f}±{np.std(ff):.3f}")

    
    if args.greedy:
        chosen, cur = [], None
        for s_ in range(args.greedy_steps):
            best = None
            for n in names:
                f = prf(Y, combine([cat(mem[m]["oof"]) for m in chosen + [n]], args.mode).argmax(1), K)["F1"]
                if best is None or f > best[0]: best = (f, n)
            if cur is not None and best[0] <= cur + 1e-6: break
            chosen.append(best[1]); cur = best[0]; print(f"  greedy step {s_ + 1}: +{best[1]} -> OOF {cur:.4f}")
        members = chosen
    else: members = names
    ens_oof = {i: combine([mem[m]["oof"][i] for m in members], args.mode) for i in ids_all}
    L = cat(ens_oof); P0 = L.argmax(1); s0 = prf(Y, P0, K)
    print(f"\nensemble of {members}\n  OOF macro-F1 = {s0['F1']:.4f}  (P={s0['P']:.4f} R={s0['R']:.4f})" + ("   [members chosen on OOF -> optimistic]" if args.greedy else ""))

    
    fold_trans = {f: fit_transitions([gold[i] for i in ids_all if fold_of[i] != f], K) for f in folds}
    all_trans = fit_transitions([gold[i] for i in ids_all], K)
    trans_oof = lambda i: fold_trans[fold_of[i]]

    final_pred_oof = P0; use_vit, best_lb, bias, use_bias, nested_ref = False, (0.0, 0.0), None, False, s0["F1"]
    if args.viterbi:
        print("\nViterbi smoothing (grid over lam = transition weight, beta = prior correction); OOF macro-F1 with T/prior fitted on the OTHER folds:")
        cache = {}
        for lam, beta in itertools.product(LAMS, BETAS):
            dec = decode_all(ids_all, ens_oof, dom_of, trans_oof, lam, beta)
            cache[(lam, beta)] = np.concatenate([dec[i] for i in ids_all])
        print("          " + "".join(f"beta={b:<6}" for b in BETAS))
        for lam in LAMS: print(f"  lam={lam:<4}" + "".join(f"{prf(Y, cache[(lam, b)], K)['F1']:<11.4f}" for b in BETAS))
        sc = {k: prf(Y, v, K)["F1"] for k, v in cache.items()}
        best_lb = max(sc, key=lambda k: (sc[k], -k[0]))
        
        nested = np.zeros(len(Y), np.int64)
        for f in folds:
            tr = fold_vec != f
            kb = max(cache, key=lambda k: (prf(Y[tr], cache[k][tr], K)["F1"], -k[0]))
            nested[fold_vec == f] = cache[kb][fold_vec == f]
        f_nested = prf(Y, nested, K)["F1"]
        print(f"  best in-sample (lam={best_lb[0]}, beta={best_lb[1]}): {sc[best_lb]:.4f} | NESTED estimate {f_nested:.4f} | no smoothing {s0['F1']:.4f}")
        use_vit = f_nested > s0["F1"] + 1e-6 or args.force
        print(f"  -> Viterbi {'APPLIED' if use_vit else 'NOT applied (nested gain <= 0)'}")
        if use_vit: final_pred_oof, nested_ref = cache[best_lb], f_nested
    if args.bias_tune:
        
        
        pb = best_lb[1] if use_vit else 0.0
        base = np.concatenate([ens_oof[i] - pb * trans_oof(i)[0] for i in ids_all])
        b_full, f_in = tune_bias(Y, base, K)
        dstart = np.concatenate([[0], np.cumsum([gold[i]["n"] for i in ids_all])])
        nested = np.zeros(len(Y), np.int64)
        for f in folds:
            tr = fold_vec != f; b_f, _ = tune_bias(Y[tr], base[tr], K)
            fi = [i for i in ids_all if fold_of[i] == f]
            if use_vit: dec = decode_all(fi, ens_oof, dom_of, trans_oof, best_lb[0], best_lb[1], b_f)
            for i in fi:
                k = ids_all.index(i); nested[dstart[k]:dstart[k + 1]] = dec[i] if use_vit else (ens_oof[i] + b_f).argmax(1)
        ref = nested_ref; f_nested = prf(Y, nested, K)["F1"]
        print(f"\nper-class bias (pipeline = {'Viterbi + bias' if use_vit else 'argmax + bias'}): in-sample argmax {f_in:.4f} | NESTED pipeline {f_nested:.4f} | same pipeline without bias (nested) {ref:.4f}\n   bias={dict(zip(labels, np.round(b_full, 2).tolist()))}")
        use_bias = f_nested > ref + 1e-6 or args.force
        print(f"  -> bias {'APPLIED' if use_bias else 'NOT applied (nested gain <= 0)'}")
        if use_bias:
            bias = b_full
            if use_vit:
                dec = decode_all(ids_all, ens_oof, dom_of, trans_oof, best_lb[0], best_lb[1], bias); final_pred_oof = np.concatenate([dec[i] for i in ids_all])
            else: final_pred_oof = (L + bias).argmax(1)

    sf = prf(Y, final_pred_oof, K); ff = fold_f1s(final_pred_oof)
    print(f"\nFINAL OOF macro-F1 = {sf['F1']:.4f}  P={sf['P']:.4f} R={sf['R']:.4f} | per-fold {np.round(ff, 3).tolist()} | CL {prf(Y[dom_vec == 0], final_pred_oof[dom_vec == 0], K)['F1']:.4f} IT {prf(Y[dom_vec == 1], final_pred_oof[dom_vec == 1], K)['F1']:.4f}")
    for i, l in enumerate(labels): print(f"    {l:24s} P={sf['per_p'][i]:.3f} R={sf['per_r'][i]:.3f} F1={sf['per_f1'][i]:.3f} support={sf['support'][i]}")
    ci = None
    if args.boot:
        rng = np.random.RandomState(0); nd = len(ids_all); bs = []
        starts = np.concatenate([[0], np.cumsum([gold[i]["n"] for i in ids_all])])
        for _ in range(args.boot):
            pick = rng.randint(nd, size=nd)
            idx = np.concatenate([np.arange(starts[k], starts[k + 1]) for k in pick]); bs.append(prf(Y[idx], final_pred_oof[idx], K)["F1"])
        ci = (float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)))
        print(f"  bootstrap over documents ({args.boot}x): 95% interval [{ci[0]:.4f}, {ci[1]:.4f}]  <- differences smaller than this are noise")

    summary = dict(members=members, mode=args.mode, oof_F1_plain=s0["F1"], oof_F1_final=sf["F1"], viterbi=dict(applied=use_vit, lam=best_lb[0], beta=best_lb[1]),
                   bias=dict(applied=use_bias, values=None if bias is None else bias.tolist()), ci95=ci, per_label_F1=dict(zip(labels, sf["per_f1"])), n_folds=nf)
    os.makedirs(args.out, exist_ok=True)
    json.dump(summary, open(os.path.join(args.out, "ensemble_cv.json"), "w"), indent=1)
    if args.no_test: return

    
    if not all(mem[m]["test"] is not None for m in members): print("some member lacks test_lp.npz -> no submission written"); return
    tids = mem[members[0]]["test_ids"]
    for m in members: assert mem[m]["test_ids"] == tids, f"{m}: test doc order differs"
    test_dom = {}
    for dn, dom in rr.SPLITS:
        for did, sents, _ in rr.iter_raw_docs(rr.load_json(os.path.join(args.data_dir, f"{dn}_test.json"))): test_dom[did] = (dom, len(sents))
    assert set(test_dom) == set(tids), "test ids differ from the test files"
    logpr, T, S = all_trans; pred = {}
    for i in tids:
        lp = combine([mem[m]["test"][i] for m in members], args.mode)
        assert len(lp) == test_dom[i][1], f"{i}: {len(lp)} rows for {test_dom[i][1]} sentences"
        if use_vit: path = viterbi(lp - best_lb[1] * logpr + (0 if bias is None else bias), T[test_dom[i][0]], S[test_dom[i][0]], best_lb[0])
        else: path = (lp + (0 if bias is None else bias)).argmax(1)
        pred[i] = [labels[k] for k in path]
    pj = os.path.join(args.out, "prediction.json"); json.dump(pred, open(pj, "w"))
    with zipfile.ZipFile(os.path.join(args.out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    cnt = np.bincount([labels.index(l) for v in pred.values() for l in v], minlength=K)
    print(f"\nwrote {args.out}/submission.zip ({len(pred)} docs, {sum(map(len, pred.values()))} sentences); predicted label counts: {dict(zip(labels, cnt.tolist()))}")


if __name__ == "__main__":
    main()

Writing ensemble_cv.py


## 1. Environment, Phase-1 files

In [4]:
import os, sys, json, time, glob, subprocess, shutil, collections
import numpy as np, pandas as pd
WORK = "/kaggle/working"; sys.path.insert(0, WORK)
def find(name, root="/kaggle/input"):
    hits = [os.path.join(d, name) for d, _, fs in os.walk(root, followlinks=True) if name in fs]
    return sorted(hits, key=os.path.getmtime)
cands = [c for n in ("model.pt", "checkpoint.pt") for c in reversed(find(n))]; print("candidates:", cands)
ed = [c for c in cands if any(t in c.lower() for t in ("encdec", "enc_dec", "encoder"))]
INIT = (ed or [None])[0]                                  
assert INIT, f"attach the encoder-decoder Phase-1 output (path must contain 'encdec'). /kaggle/input has: {os.listdir('/kaggle/input')}"
toks = find("bpe_tokenizer.json"); TOK = next((t for t in toks if os.path.dirname(t) == os.path.dirname(INIT)), None)
assert TOK, "the encoder-decoder notebook trained ITS OWN tokenizer: attach bpe_tokenizer.json from the same output folder as model.pt (a different tokenizer would give garbage)"
print("INIT:", INIT, "\nTOK :", TOK)
import torch; print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

candidates: ['/kaggle/input/datasets/akshat1206/enco-deco-results/cs781_phase1/encdec_rope_geglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt', '/kaggle/input/datasets/akshat1206/enco-deco-results/cs781_phase1/encdec_rope_geglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/checkpoint.pt']
INIT: /kaggle/input/datasets/akshat1206/enco-deco-results/cs781_phase1/encdec_rope_geglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt 
TOK : /kaggle/input/datasets/akshat1206/enco-deco-results/cs781_phase1/encdec_rope_geglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/bpe_tokenizer.json
GPUs: 2 ['Tesla T4', 'Tesla T4']


## 2. Data (Hugging Face `Exploration-Lab/CS-781-Capstone`)

In [5]:
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HuggingFace")
DATA = f"{WORK}/p3data"; os.makedirs(DATA, exist_ok=True)
repo_files = [f for f in list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=hf_token) if "Phase_3" in f]
print("\n".join(repo_files))
WANT = ["CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json", "CL_test.json", "IT_test.json", "label_vocab.json"]
for w in WANT:
    hit = [f for f in repo_files if os.path.basename(f) == w]
    assert hit, f"{w} not found in the repo listing above"
    if not os.path.exists(f"{DATA}/{w}"):
        shutil.copy(hf_hub_download("Exploration-Lab/CS-781-Capstone", hit[0], repo_type="dataset", token=hf_token), f"{DATA}/{w}")
print(sorted(os.listdir(DATA)))

Phase_3_RR/Test/CL_test.json
Phase_3_RR/Test/IT_test.json
Phase_3_RR/Train/CL_dev.json
Phase_3_RR/Train/CL_train.json
Phase_3_RR/Train/IT_dev.json
Phase_3_RR/Train/IT_train.json
Phase_3_RR/Train/label_vocab.json


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

['CL_dev.json', 'CL_test.json', 'CL_train.json', 'IT_dev.json', 'IT_test.json', 'IT_train.json', 'label_vocab.json']


## 3. Data analysis (material for the report) + schema check

In [6]:
import rr
from collections import Counter
labels, label2idx = rr.load_vocab(rr.load_json(f"{DATA}/label_vocab.json")); print(labels)
raw = {k: list(rr.iter_raw_docs(rr.load_json(f"{DATA}/{k}.json"))) for k in ["CL_train","CL_dev","IT_train","IT_dev","CL_test","IT_test"]}
first = rr.load_json(f"{DATA}/CL_train.json")
print("top-level type:", type(first).__name__, "| first item keys:", list(first[0].keys()) if isinstance(first, list) else list(next(iter(first.values())).keys()) if isinstance(next(iter(first.values())), dict) else "id -> list")
rows = []
for k, docs in raw.items():
    n_s = [len(s) for _, s, _ in docs]
    rows.append(dict(split=k, docs=len(docs), sentences=sum(n_s), mean_sent_per_doc=round(np.mean(n_s), 1), max_sent_per_doc=max(n_s),
                     has_labels=docs[0][2] is not None, mean_words_per_sentence=round(np.mean([len(x.split()) for _, s, _ in docs for x in s]), 1)))
display(pd.DataFrame(rows))
dist = pd.DataFrame({k: Counter(l if isinstance(l, str) else labels[l] for _, _, ls in raw[k] for l in ls) for k in ["CL_train","CL_dev","IT_train","IT_dev"]}).reindex(labels).fillna(0).astype(int)
dist["train_total"] = dist.CL_train + dist.IT_train; dist["train_%"] = (100 * dist.train_total / dist.train_total.sum()).round(1)
display(dist)

for dn in ("CL", "IT"):
    json.dump({did: [l if isinstance(l, str) else labels[l] for l in ls] for did, _, ls in raw[f"{dn}_dev"]}, open(f"{DATA}/truth_{dn}_dev.json", "w"))
assert not set(d for d, _, _ in raw["CL_dev"]) & set(d for d, _, _ in raw["IT_dev"]), "CL/IT dev ids overlap"

['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
top-level type: list | first item keys: ['id', 'text', 'labels']


,split,docs,sentences,mean_sent_per_doc,max_sent_per_doc,has_labels,mean_words_per_sentence
0,CL_train,40,10593,264.8,1816,True,32.7
1,CL_dev,5,1396,279.2,848,True,33.7
2,IT_train,40,6252,156.3,442,True,35.3
3,IT_dev,5,746,149.2,382,True,32.3
4,CL_test,5,1339,267.8,942,False,33.2
5,IT_test,5,858,171.6,420,False,37.6


,CL_train,CL_dev,IT_train,IT_dev,train_total,train_%
Fact,2979,249,1119,195,4098,24.3
Issue,17,3,90,12,107,0.6
ArgumentPetitioner,976,344,637,56,1613,9.6
ArgumentRespondent,996,236,352,36,1348,8.0
Statute,367,58,732,62,1099,6.5
Dissent,284,0,0,0,284,1.7
PrecedentReliedUpon,1305,142,640,112,1945,11.5
PrecedentNotReliedUpon,209,0,360,50,569,3.4
PrecedentOverruled,0,0,3,0,3,0.0
RulingByLowerCourt,438,56,436,46,874,5.2


## 4. Launcher (one process per GPU via `CUDA_VISIBLE_DEVICES`)

In [7]:
os.makedirs(f"{WORK}/runs", exist_ok=True)
BUDGET = 60   

def cmd(name, gpu, extra, epochs, budget=None, teacher_run=None):
    out = f"{WORK}/runs/{name}"; os.makedirs(out, exist_ok=True)
    t = f"--teacher {WORK}/runs/{teacher_run}/best.pt" if teacher_run else ""
    return (f"PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} python rr_ed.py --data_dir {DATA} --tokenizer {TOK} --init {INIT} --out {out} "
            f"--epochs {epochs} --time_budget_min {budget or BUDGET} {t} {extra} > {out}/log.txt 2>&1")

def launch(gpu, jobs):
    """jobs = [(name, extra_args, epochs, teacher_run_or_None)] run one after another on `gpu`."""
    p = subprocess.Popen(" ; ".join(cmd(n, gpu, e, ep, teacher_run=t) for n, e, ep, t in jobs), shell=True)
    return p, [j[0] for j in jobs]

def wait(procs, every=120):
    while any(p.poll() is None for p, _ in procs):
        time.sleep(every)
        for p, names in procs:
            logs = [l for n in names for l in glob.glob(f"{WORK}/runs/{n}/log.txt")]
            if not logs: continue
            lg = max(logs, key=os.path.getmtime); tail = open(lg).read().strip().splitlines()[-1:]
            print(f"[{lg.split('/')[-2]}] {'running' if p.poll() is None else 'done'} | {tail[0][:130] if tail else ''}")
    for p, names in procs:
        for n in names:
            ok = os.path.exists(f"{WORK}/runs/{n}/results.json")
            print(f"== {n}: {'OK' if ok else 'FAILED  <-- see runs/' + n + '/log.txt'}")

def table(pattern="*"):
    rows = []
    for d in sorted(glob.glob(f"{WORK}/runs/{pattern}/results.json")):
        x = json.load(open(d)); pd_ = x.get("dev_per_domain_F1", {})
        rows.append(dict(run=d.split("/")[-2], method=x["method"], attn=x["attn"], dev_F1=round(x["dev_macro_F1"], 4), P=round(x["dev_macro_P"], 4), R=round(x["dev_macro_R"], 4),
                         CL=round(pd_.get("CL", float("nan")), 4), IT=round(pd_.get("IT", float("nan")), 4), trainable_M=round(x["params_trainable"] / 1e6, 3),
                         train_min=round(x["train_minutes"], 1), steps=f"{x['steps']}/{x['planned_steps']}", best_step=x["best_step"], gpu_GB=round(x["peak_gpu_gb"], 1),
                         infer_tok_s=int(x["infer_tokens_per_s"])))
    return pd.DataFrame(rows).sort_values("dev_F1", ascending=False) if rows else pd.DataFrame()

def move_smoke():
    os.makedirs(f"{WORK}/smoke", exist_ok=True)
    for d in glob.glob(f"{WORK}/runs/smoke_*"): shutil.move(d, f"{WORK}/smoke/" + os.path.basename(d))

## 5. YOUR KNOBS (CV edition)
Untuned starting values — **you** must choose / justify them. `NF` folds, `SEEDS` = independent repeats of every fold run (a second seed halves the training noise of an ensemble but doubles the cost),
`TTA` = number of shifted window partitions averaged at test time (1 = off, max 6).

In [8]:
NF, SEEDS, TTA = 5, [1], 3
EP   = dict(full=5, kd=6, lora=8, lorma=8, adalora=8, ia3=15, prompt=15)
LR   = dict(full=None, kd=None, lora=None, lorma=None, adalora=None, ia3=None, prompt=None)
RANK = 16
ATTN = "bidir"
ED_MODE = "enc"      
COMMON = f"--cw_pow 0.5 --window {512 if ED_MODE == 'enc' else 400} --max_sent 128 --bs 8 --ed_mode {ED_MODE}"
def X(m, extra=""):
    lr = f"--lr {LR[m]}" if LR[m] else ""
    return f"--method {m} --attn {ATTN} {lr} {COMMON} {extra}"

## 6. The CV launcher
`cv_round(method, seed)` queues the NF fold runs of one method alternately on GPU0 / GPU1 (runs `<prefix>cv_<method>_s<seed>_f<k>`). `cv_status()` lists what is finished.
The KD student of fold *k* uses the full-FT teacher of the **same** fold (otherwise the teacher would have seen the held-out documents and the OOF score would leak).

In [9]:
FOLD = "--n_folds {NF} --fold {f} --fold_seed 0 --tta {TTA} --seed {seed}"
def cv_round(method, seed, extra="", only_missing=True):
    qs = [[], []]; k = 0
    for f in range(NF):
        name = f"ed_cv_{method}_s{seed}_f{f}"
        if only_missing and os.path.exists(f"{WORK}/runs/{name}/results.json"): continue
        args = X(method, extra + " " + FOLD.format(NF=NF, f=f, TTA=TTA, seed=seed))
        teacher = f"ed_cv_full_s{seed}_f{f}" if method == "kd" else None
        qs[k % 2].append((name, args, EP[method], teacher)); k += 1
    ps = [launch(g, q) for g, q in enumerate(qs) if q]
    if ps: wait(ps)
    return cv_status(f"ed_cv_{method}_s{seed}")
def cv_status(prefix=None):
    rows = []
    for d in sorted(glob.glob(f"{WORK}/runs/{prefix or P + 'cv_'}*_f[0-9]*/results.json")):
        x = json.load(open(d)); rows.append(dict(run=d.split("/")[-2], held_out_F1=round(x["dev_macro_F1"], 4), single_pass_F1=round(x.get("dev_macro_F1_single_pass", float("nan")), 4),
                                               min=round(x["train_minutes"], 1), steps=f"{x['steps']}/{x['planned_steps']}"))
    return pd.DataFrame(rows)

## 7. Smoke test of the CV path (a few minutes)

In [10]:
SM = "--max_train_docs 12 --no_test --n_folds 3 --fold 0 --tta 2"
jobs0 = [("smoke_full", f"--method full {SM}", 1, None), ("smoke_kd", f"--method kd {SM}", 1, "smoke_full"), ("smoke_lora", f"--method lora {SM}", 1, None)]
jobs1 = [("smoke_lorma", f"--method lorma {SM}", 1, None), ("smoke_adalora", f"--method adalora {SM}", 1, None)]
jobs1 = [(n, a.replace("--method", "--method", 1) + " " + COMMON + (" --attn " + ATTN if not False else ""), e, t) for n, a, e, t in jobs1]
jobs0 = [(n, a + " " + COMMON + (" --attn " + ATTN if not False else ""), e, t) for n, a, e, t in jobs0]
pA = launch(0, jobs0); pB = launch(1, jobs1); wait([pA, pB], every=30)
display(table("smoke_*")); move_smoke()

[smoke_kd] running |   [eval final] step=14 dev macro-F1=0.0242 P=0.0580 R=0.0858
[smoke_adalora] running |   AdaLoRA: 42 modules, budget 1008 -> 672 triplets, tinit=40 tfinal=4
[smoke_lora] done | == done in 0.1 min | dev macro-F1 = 0.0084
[smoke_adalora] done | == done in 0.1 min | dev macro-F1 = 0.0084
== smoke_full: OK
== smoke_kd: OK
== smoke_lora: OK
== smoke_lorma: OK
== smoke_adalora: OK


,run,method,attn,dev_F1,P,R,CL,IT,trainable_M,train_min,steps,best_step,gpu_GB,infer_tok_s
2,smoke_kd,kd,bidir,0.0241,0.0580,0.0858,0.0081,0.0321,27.355,0.0,14/14,14,1.3,106794
4,smoke_lorma,lorma,bidir,0.0087,0.0046,0.0769,0.0081,0.0090,1.607,0.0,14/14,14,1.4,39853
0,smoke_adalora,adalora,bidir,0.0084,0.0045,0.0769,0.0079,0.0087,2.130,0.1,14/14,14,1.5,37672
1,smoke_full,full,bidir,0.0084,0.0045,0.0769,0.0079,0.0087,47.019,0.0,14/14,14,1.9,56166
3,smoke_lora,lora,bidir,0.0084,0.0045,0.0769,0.0079,0.0087,1.423,0.0,14/14,14,1.5,38154


## 8. Round 1 — full fine-tuning, 5 folds (baseline **and** the teachers for KD)

In [11]:
for seed in SEEDS: display(cv_round("full", seed))

[ed_cv_full_s1_f0] running |   step 750/1255 ep 3 loss 1.2523 lr 2.02e-05 1.9min
[ed_cv_full_s1_f1] running |   step 800/1545 ep 3 loss 1.2532 lr 2.63e-05 1.9min
[ed_cv_full_s1_f0] running |   [eval final] step=1255 dev macro-F1=0.2623 P=0.2697 R=0.2813
[ed_cv_full_s1_f1] running |   [eval final] step=1545 dev macro-F1=0.3037 P=0.3082 R=0.3140
[ed_cv_full_s1_f2] running |   step 650/1410 ep 3 loss 1.4154 lr 3.13e-05 1.7min
[ed_cv_full_s1_f3] running |   step 650/1420 ep 3 loss 1.2590 lr 3.15e-05 1.7min
[ed_cv_full_s1_f2] running |   [eval final] step=1410 dev macro-F1=0.3105 P=0.3104 R=0.3269
[ed_cv_full_s1_f3] running |   [eval final] step=1420 dev macro-F1=0.3334 P=0.3425 R=0.3374
[ed_cv_full_s1_f4] running |   step 550/1475 ep 2 loss 1.4553 lr 3.80e-05 1.3min
[ed_cv_full_s1_f3] done | == done in 4.2 min | dev macro-F1 = 0.3325
[ed_cv_full_s1_f4] running |   step 1350/1475 ep 5 loss 0.8567 lr 1.03e-06 3.3min
[ed_cv_full_s1_f3] done | == done in 4.2 min | dev macro-F1 = 0.3325
[ed_cv_

,run,held_out_F1,single_pass_F1,min,steps
0,ed_cv_full_s1_f0,0.2617,0.2623,3.3,1255/1255
1,ed_cv_full_s1_f1,0.3043,0.3037,3.7,1545/1545
2,ed_cv_full_s1_f2,0.3104,0.3105,3.6,1410/1410
3,ed_cv_full_s1_f3,0.3325,0.3334,3.6,1420/1420
4,ed_cv_full_s1_f4,0.2829,0.2809,3.6,1475/1475


## 9. Round 2 — PEFT methods, 5 folds each (LoRA · LoRMA₊ · AdaLoRA)
Longest part (≈ 3 × NF runs). If time is short: cut `SEEDS`/`NF` or run one method per Kaggle session — finished folds are skipped on re-run (`only_missing=True`), and an incomplete method is ignored by `ensemble_cv.py`.

In [12]:
for seed in SEEDS:
    display(cv_round("lora", seed, f"--rank {RANK} --alpha {2*RANK}"))
    display(cv_round("lorma", seed, f"--rank {RANK} --alpha {2*RANK}"))
    display(cv_round("adalora", seed, "--ada_init_r 24 --ada_target_r 16 --alpha_ada 48"))

[ed_cv_lora_s1_f0] running |   step 650/2008 ep 3 loss 1.6179 lr 8.09e-04 1.9min
[ed_cv_lora_s1_f1] running |   step 800/2472 ep 3 loss 1.5711 lr 8.01e-04 1.9min
[ed_cv_lora_s1_f0] running |   step 1300/2008 ep 6 loss 1.3205 lr 3.04e-04 3.9min
[ed_cv_lora_s1_f1] running |   step 1600/2472 ep 6 loss 1.1412 lr 2.99e-04 3.9min
[ed_cv_lora_s1_f0] running |   step 2000/2008 ep 8 loss 1.1998 lr 5.49e-08 5.9min
[ed_cv_lora_s1_f1] running |   step 2400/2472 ep 8 loss 1.1079 lr 2.34e-06 5.9min
[ed_cv_lora_s1_f2] running |   step 300/2256 ep 2 loss 2.0119 lr 9.79e-04 0.8min
[ed_cv_lora_s1_f3] running |   step 450/2272 ep 2 loss 1.7995 lr 9.38e-04 1.2min
[ed_cv_lora_s1_f2] running |   step 1050/2256 ep 4 loss 1.4309 lr 5.93e-04 2.8min
[ed_cv_lora_s1_f3] running |   step 1150/2272 ep 5 loss 1.3052 lr 5.27e-04 3.1min
[ed_cv_lora_s1_f2] running |   step 1750/2256 ep 7 loss 1.1609 lr 1.30e-04 4.8min
[ed_cv_lora_s1_f3] running |   step 1900/2272 ep 7 loss 1.1140 lr 7.10e-05 5.1min
[ed_cv_lora_s1_f2] r

,run,held_out_F1,single_pass_F1,min,steps
0,ed_cv_lora_s1_f0,0.3114,0.3115,5.9,2008/2008
1,ed_cv_lora_s1_f1,0.3732,0.3692,6.1,2472/2472
2,ed_cv_lora_s1_f2,0.3464,0.3469,6.1,2256/2256
3,ed_cv_lora_s1_f3,0.3341,0.3355,6.1,2272/2272
4,ed_cv_lora_s1_f4,0.3148,0.3136,6.2,2360/2360


[ed_cv_lorma_s1_f0] running |   step 750/2008 ep 3 loss 1.5836 lr 7.41e-04 1.9min
[ed_cv_lorma_s1_f1] running |   step 900/2472 ep 3 loss 1.6689 lr 7.45e-04 1.9min
[ed_cv_lorma_s1_f0] running |   step 1450/2008 ep 6 loss 1.3427 lr 1.97e-04 3.9min
[ed_cv_lorma_s1_f1] running |   step 1750/2472 ep 6 loss 1.4382 lr 2.12e-04 3.8min
[ed_cv_lorma_s1_f0] running |   [eval final] step=2008 dev macro-F1=0.3277 P=0.3252 R=0.3576
[ed_cv_lorma_s1_f1] running |     None                     P=0.449 R=0.349 F1=0.393 support=63
[ed_cv_lorma_s1_f2] running |   step 550/2256 ep 2 loss 1.8608 lr 8.97e-04 1.3min
[ed_cv_lorma_s1_f3] running |   step 700/2272 ep 3 loss 1.6921 lr 8.24e-04 1.7min
[ed_cv_lorma_s1_f2] running |   step 1300/2256 ep 5 loss 1.4681 lr 4.12e-04 3.2min
[ed_cv_lorma_s1_f3] running |   step 1500/2272 ep 6 loss 1.3582 lr 2.81e-04 3.7min
[ed_cv_lorma_s1_f2] running |   step 2100/2256 ep 8 loss 1.2914 lr 1.30e-05 5.3min
[ed_cv_lorma_s1_f3] running |   [eval final] step=2272 dev macro-F1=0

,run,held_out_F1,single_pass_F1,min,steps
0,ed_cv_lorma_s1_f0,0.3288,0.3277,5.4,2008/2008
1,ed_cv_lorma_s1_f1,0.3614,0.3582,5.5,2472/2472
2,ed_cv_lorma_s1_f2,0.3177,0.3185,5.6,2256/2256
3,ed_cv_lorma_s1_f3,0.3268,0.3260,5.6,2272/2272
4,ed_cv_lorma_s1_f4,0.3010,0.3008,5.6,2360/2360


[ed_cv_adalora_s1_f0] running |   [eval mid] step=502 dev macro-F1=0.2313 P=0.2415 R=0.2604
[ed_cv_adalora_s1_f1] running |   step 600/2472 ep 2 loss 1.8160 lr 8.95e-04 1.8min
[ed_cv_adalora_s1_f0] running |   step 1050/2008 ep 5 loss 1.5713 lr 5.04e-04 3.9min
[ed_cv_adalora_s1_f1] running |   [eval mid] step=1236 dev macro-F1=0.2993 P=0.3366 R=0.3037
[ed_cv_adalora_s1_f0] running |   step 1600/2008 ep 7 loss 1.3684 lr 1.09e-04 5.8min
[ed_cv_adalora_s1_f1] running |   [eval mid] step=1854 dev macro-F1=0.3315 P=0.4116 R=0.3372
[ed_cv_adalora_s1_f0] running |   [eval final] step=2008 dev macro-F1=0.2919 P=0.2985 R=0.3146
[ed_cv_adalora_s1_f1] running |   [eval final] step=2472 dev macro-F1=0.3333 P=0.4109 R=0.3365
[ed_cv_adalora_s1_f2] running |   step 450/2256 ep 2 loss 1.8018 lr 9.37e-04 1.5min
[ed_cv_adalora_s1_f3] running |   step 500/2272 ep 2 loss 1.7662 lr 9.19e-04 1.6min
[ed_cv_adalora_s1_f2] running |   step 1050/2256 ep 4 loss 1.5932 lr 5.93e-04 3.5min
[ed_cv_adalora_s1_f3] run

,run,held_out_F1,single_pass_F1,min,steps
0,ed_cv_adalora_s1_f0,0.2925,0.2919,7.1,2008/2008
1,ed_cv_adalora_s1_f1,0.3344,0.3333,7.6,2472/2472
2,ed_cv_adalora_s1_f2,0.3301,0.3314,7.4,2256/2256
3,ed_cv_adalora_s1_f3,0.3148,0.3145,7.4,2272/2272
4,ed_cv_adalora_s1_f4,0.2905,0.2914,7.5,2360/2360


## 10. Round 3 — KD (6-layer student per fold, teacher = same-fold full FT) and (IA)³

In [13]:
for seed in SEEDS:
    display(cv_round("kd", seed, "--student_layers 3 --kd_T 2.0 --kd_alpha 0.5"))
    display(cv_round("ia3", seed))

[ed_cv_kd_s1_f0] running |   step 950/1506 ep 4 loss 0.4762 lr 3.40e-05 1.9min
[ed_cv_kd_s1_f1] running |   step 1100/1854 ep 4 loss 0.4628 lr 3.92e-05 1.9min
[ed_cv_kd_s1_f2] running |   step 150/1692 ep 1 loss 1.4607 lr 9.98e-05 0.3min
[ed_cv_kd_s1_f3] running |   step 150/1704 ep 1 loss 1.3956 lr 9.98e-05 0.3min
[ed_cv_kd_s1_f2] running |   step 1150/1692 ep 5 loss 0.4551 lr 2.61e-05 2.3min
[ed_cv_kd_s1_f3] running |   step 1200/1704 ep 5 loss 0.4131 lr 2.25e-05 2.3min
[ed_cv_kd_s1_f4] running |   step 300/1770 ep 2 loss 1.0621 lr 9.65e-05 0.6min
[ed_cv_kd_s1_f3] done | == done in 3.6 min | dev macro-F1 = 0.3478
[ed_cv_kd_s1_f4] running |   step 1350/1770 ep 5 loss 0.3832 lr 1.49e-05 2.6min
[ed_cv_kd_s1_f3] done | == done in 3.6 min | dev macro-F1 = 0.3478
[ed_cv_kd_s1_f4] done | == done in 3.7 min | dev macro-F1 = 0.2994
[ed_cv_kd_s1_f3] done | == done in 3.6 min | dev macro-F1 = 0.3478
== ed_cv_kd_s1_f0: OK
== ed_cv_kd_s1_f2: OK
== ed_cv_kd_s1_f4: OK
== ed_cv_kd_s1_f1: OK
== ed_cv

,run,held_out_F1,single_pass_F1,min,steps
0,ed_cv_kd_s1_f0,0.2849,0.2841,3.1,1506/1506
1,ed_cv_kd_s1_f1,0.3347,0.3364,3.3,1854/1854
2,ed_cv_kd_s1_f2,0.3418,0.3426,3.3,1692/1692
3,ed_cv_kd_s1_f3,0.3478,0.3490,3.3,1704/1704
4,ed_cv_kd_s1_f4,0.2994,0.2989,3.4,1770/1770


[ed_cv_ia3_s1_f0] running |   [eval mid] step=753 dev macro-F1=0.2505 P=0.2680 R=0.2765
[ed_cv_ia3_s1_f1] running |   step 950/4635 ep 4 loss 1.6426 lr 9.16e-03 1.9min
[ed_cv_ia3_s1_f0] running |   step 1550/3765 ep 7 loss 1.4939 lr 6.61e-03 3.9min
[ed_cv_ia3_s1_f1] running |   step 1950/4635 ep 7 loss 1.4586 lr 6.43e-03 3.9min
[ed_cv_ia3_s1_f0] running |   step 2350/3765 ep 10 loss 1.4044 lr 3.25e-03 5.9min
[ed_cv_ia3_s1_f1] running |   step 2900/4635 ep 10 loss 1.4509 lr 3.20e-03 5.9min
[ed_cv_ia3_s1_f0] running |   step 3150/3765 ep 13 loss 1.4021 lr 6.81e-04 7.9min
[ed_cv_ia3_s1_f1] running |   step 3800/4635 ep 13 loss 1.3650 lr 8.15e-04 7.9min
[ed_cv_ia3_s1_f0] running |   [eval final] step=3765 dev macro-F1=0.2792 P=0.2768 R=0.3032
[ed_cv_ia3_s1_f1] running |   [eval final] step=4635 dev macro-F1=0.3383 P=0.3845 R=0.3471
[ed_cv_ia3_s1_f2] running |   step 650/4230 ep 3 loss 1.7341 lr 9.57e-03 1.5min
[ed_cv_ia3_s1_f3] running |   step 750/4260 ep 3 loss 1.6487 lr 9.41e-03 1.7min


,run,held_out_F1,single_pass_F1,min,steps
0,ed_cv_ia3_s1_f0,0.2777,0.2792,9.4,3765/3765
1,ed_cv_ia3_s1_f1,0.3382,0.3383,9.6,4635/4635
2,ed_cv_ia3_s1_f2,0.3262,0.3295,9.6,4230/4230
3,ed_cv_ia3_s1_f3,0.3078,0.3075,9.6,4260/4260
4,ed_cv_ia3_s1_f4,0.2955,0.2965,9.7,4425/4425


## 11. Prompt tuning (optional)
Needed only if you want prompt/prefix as your “prompt or prefix” method. In the plain notebook it was the weakest method and hurt ensembles — it is excluded from the ensembles below either way.

In [14]:
RUN_PROMPT = False
if RUN_PROMPT:
    for seed in SEEDS: display(cv_round("prompt", seed, "--n_prompt 32"))

## 12. OOF evaluation, ensembles, submissions
Prefixes are `<prefix>cv_<method>_s<seed>`. `ensemble_cv.py` prints: per-member OOF macro-F1 (+ fold mean±sd), the ensemble, the Viterbi grid with its **nested** estimate, the bias check, a bootstrap interval, then writes `submission.zip`.
**Cross-model mixing:** attach the other notebooks' *outputs* and fill `OTHER` ({tag: runs folder}); their fold runs are copied in (all models use the identical fold split, which the script verifies).

In [15]:
OTHER = {}      
for tag, folder in OTHER.items():
    for src in sorted(glob.glob(f"{folder}/*cv_*_f[0-9]*")):
        n = os.path.basename(src)
        if n.startswith("smoke") or not os.path.exists(f"{src}/dev_lp.npz"): continue
        dst = f"{WORK}/runs/" + (n if n.startswith(tag + "_") else f"{tag}_{n}"); os.makedirs(dst, exist_ok=True)
        for f in ["dev_lp.npz", "dev_ids.json", "test_lp.npz", "test_ids.json", "results.json"]:
            if os.path.exists(f"{src}/{f}"): shutil.copy(f"{src}/{f}", f"{dst}/{f}")
        print("imported", dst)

In [16]:
os.makedirs(f"{WORK}/submissions", exist_ok=True)
def cvens(out, members, extra=""):
    r = subprocess.run(f"python ensemble_cv.py --runs_dir {WORK}/runs --data_dir {DATA} --out {WORK}/submissions/{out} {members} {extra}", shell=True, capture_output=True, text=True)
    print(r.stdout[-6000:] + r.stderr[-1500:])
    if os.path.exists(f"{WORK}/submissions/{out}/submission.zip"): shutil.copy(f"{WORK}/submissions/{out}/submission.zip", f"{WORK}/submissions/{out}.zip")
S1 = SEEDS[0]; pre = lambda m: f"ed_cv_{m}_s{S1}"

for m in ["full", "lora", "lorma", "adalora", "ia3", "kd"]:
    cvens(f"ed_cv_{m}", f"--members {pre(m)}", "--viterbi")

90 labelled docs (CL=45, IT=45), 18987 sentences
5 folds, docs per fold: [18, 18, 18, 18, 18]

single members (OOF = every labelled doc predicted by a model that never saw it):
  member                          OOF F1      P      R     CL     IT   fold-mean±sd
  ed_cv_full_s1                   0.3044  0.304  0.309  0.292  0.287   0.2984±0.024

ensemble of ['ed_cv_full_s1']
  OOF macro-F1 = 0.3044  (P=0.3037 R=0.3090)

Viterbi smoothing (grid over lam = transition weight, beta = prior correction); OOF macro-F1 with T/prior fitted on the OTHER folds:
          beta=0.0   beta=0.5   beta=1.0   
  lam=0.0 0.3044     0.2951     0.2605     
  lam=0.250.3052     0.3088     0.2719     
  lam=0.5 0.3026     0.3146     0.2821     
  lam=0.750.2989     0.3177     0.2875     
  lam=1.0 0.2950     0.3162     0.2887     
  lam=1.5 0.2880     0.3119     0.2896     
  best in-sample (lam=0.75, beta=0.5): 0.3177 | NESTED estimate 0.3177 | no smoothing 0.3044
  -> Viterbi APPLIED

FINAL OOF macro-F1 = 0

In [17]:

cvens("ed_cv_ens_peft", "--members " + ",".join(pre(m) for m in ["lora", "lorma", "adalora"]), "--viterbi")
cvens("ed_cv_ens_all",  "--members " + ",".join(pre(m) for m in ["full", "lora", "lorma", "adalora", "kd"]), "--viterbi")
cvens("ed_cv_ens_greedy", "--all --greedy", "--viterbi")          

90 labelled docs (CL=45, IT=45), 18987 sentences
5 folds, docs per fold: [18, 18, 18, 18, 18]

single members (OOF = every labelled doc predicted by a model that never saw it):
  member                          OOF F1      P      R     CL     IT   fold-mean±sd
  ed_cv_lora_s1                   0.3379  0.337  0.345  0.332  0.309   0.3360±0.023
  ed_cv_lorma_s1                  0.3332  0.343  0.337  0.321  0.308   0.3271±0.020
  ed_cv_adalora_s1                0.3168  0.318  0.324  0.312  0.289   0.3125±0.018

ensemble of ['ed_cv_lora_s1', 'ed_cv_lorma_s1', 'ed_cv_adalora_s1']
  OOF macro-F1 = 0.3435  (P=0.3590 R=0.3473)

Viterbi smoothing (grid over lam = transition weight, beta = prior correction); OOF macro-F1 with T/prior fitted on the OTHER folds:
          beta=0.0   beta=0.5   beta=1.0   
  lam=0.0 0.3435     0.3243     0.2660     
  lam=0.250.3429     0.3372     0.2845     
  lam=0.5 0.3382     0.3519     0.2952     
  lam=0.750.3305     0.3540     0.3009     
  lam=1.0 0.3236   

## 13. Collect
Everything in `/kaggle/working/submissions/*.zip` has `prediction.json` at the zip root (CL + IT merged). Upload a few (≤ 10 per day). **Compare the leaderboard with the OOF numbers**: if the ranking agrees, the OOF is trustworthy; if not, believe the leaderboard and say so in your analysis.

In [18]:
import zipfile
for z in sorted(glob.glob(f"{WORK}/submissions/*.zip")):
    with zipfile.ZipFile(z) as f: p = json.load(f.open("prediction.json"))
    print(f"{os.path.basename(z):30s} files={zipfile.ZipFile(z).namelist()} docs={len(p)} sentences={sum(map(len, p.values()))}")

ed_cv_adalora.zip              files=['prediction.json'] docs=10 sentences=2197
ed_cv_ens_all.zip              files=['prediction.json'] docs=10 sentences=2197
ed_cv_ens_greedy.zip           files=['prediction.json'] docs=10 sentences=2197
ed_cv_ens_peft.zip             files=['prediction.json'] docs=10 sentences=2197
ed_cv_full.zip                 files=['prediction.json'] docs=10 sentences=2197
ed_cv_ia3.zip                  files=['prediction.json'] docs=10 sentences=2197
ed_cv_kd.zip                   files=['prediction.json'] docs=10 sentences=2197
ed_cv_lora.zip                 files=['prediction.json'] docs=10 sentences=2197
ed_cv_lorma.zip                files=['prediction.json'] docs=10 sentences=2197
